# 11 · Anonymous-user inference and content-to-ALS comparison

## Purpose

This notebook investigates whether the existing CineMatch+ ALS recommender can be improved for **new anonymous visitors** who provide a small set of liked movies.

It compares two research directions without changing the deployed backend:

1. **LightFM fold-in:** infer a new LightFM user vector from selected movie vectors (centroid and deterministic ridge-style fold-in).
2. **Content-to-ALS factor imputation:** learn a mapping from content-v1 movie features to well-estimated ALS item factors, predict factors for warm-tail movies, and reuse the existing ALS user fold-in.

It also includes controls to test whether the extra modelling is necessary:

- ALS-only candidate ranking/gating.
- LightFM Identity-only scoring.
- LightFM Content-v1 scoring.

> **Notebook 10 remains the frozen known-user experiment.** Reuse its data definitions, LightFM feature construction, ALS half-step and evaluation conventions, but do not append this new work to Notebook 10. This notebook is a separate anonymous-user experiment so its protocol and results remain distinguishable.

## Evaluation design

- Use the existing `sample_10pct` **train and validation** splits only. Do not load or inspect `test` in this notebook.
- Simulate pseudo-new users by holding out complete user IDs from global model fitting.
- For each held-out user, sample 3, 5, 10 or 20 historical positive movies as the visitor's profile. Evaluate against that user's post-2016 validation positives.
- Compare uniform support sampling with popularity-biased support sampling.
- Recompute item popularity and warm-tail masks inside each fold using only the non-held-out model-training users.
- Exclude only the selected profile movies in the primary simulation; full-history exclusion is a sensitivity analysis.
- Use user-grouped out-of-fold evaluation. Any hyperparameter/gate choice is made on a development cohort and frozen before a separate confirmation cohort is scored.
- Use rank percentiles / matched injection rates rather than assuming the historical absolute rank 500 transfers to a different inference method.

## Important limitation

A 10% sample may not contain enough warm-tail pseudo-users for a precise independent confirmation result. The first phase below is an **eligibility and power audit only**. It deliberately does not fit models. If the sample is underpowered, we will plan an external confirmation cohort from the remaining MovieLens users before doing expensive cross-validation.

## Candidate methods (planned)

| Arm | New-user representation | Candidate scoring |
|---|---|---|
| ALS baseline | Existing implicit-ALS fold-in on selected item factors | ALS item factors |
| ALS-only gate control | Same ALS fold-in | ALS item factors, warm-tail candidate gate |
| LightFM Identity centroid | Mean of selected effective item vectors | LightFM identity item vectors + item bias |
| LightFM Identity fold-in | Closed-form implicit/ridge-style fold-in | LightFM identity item vectors + item bias |
| LightFM Content-v1 centroid | Mean of selected effective item vectors | Content-v1 item vectors + item bias |
| LightFM Content-v1 fold-in | Closed-form implicit/ridge-style fold-in | Content-v1 item vectors + item bias |
| Content-imputed ALS | Fold-in using ALS factors, with warm-tail item factors predicted from content-v1 | Imputed ALS item factors |

The WARP-style per-request optimization method is intentionally **deferred**. It is not part of the first comparison.

## 1 · Environment and project loading

This setup intentionally loads only `train`, `val`, user/item mappings and the frozen content-v1 feature matrix. It does not load `test.parquet`. The project-root discovery and feature paths follow Notebook 10.

In [1]:
# Setup and dependency checks — mirrors Notebook 10's project discovery.
import os
import sys
import time
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import csr_matrix
from sklearn.model_selection import KFold

try:
    from lightfm import LightFM
except ImportError as exc:
    raise ImportError(
        "LightFM is required for the LightFM comparison arms. "
        "Activate the same project environment used by Notebook 10 and install LightFM there."
    ) from exc

_START = Path.cwd().resolve()
PROJECT_ROOT = None
for _p in [_START, *_START.parents]:
    if (_p / "scripts" / "recutils.py").exists() and (_p / "data").exists():
        PROJECT_ROOT = _p
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root containing scripts/recutils.py and data/. "
        "Place this notebook in the project's notebooks/ folder and open it from there."
    )

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
if NOTEBOOK_DIR.exists() and Path.cwd().resolve() != NOTEBOOK_DIR.resolve():
    os.chdir(NOTEBOOK_DIR)

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

NAME = "sample_10pct"
REL = 4.0
SEED = 42
CUTOFF_TS = int(pd.Timestamp("2016-01-01 00:00:00", tz="UTC").timestamp())
PROFILE_SIZES = [3, 5, 10, 20]
N_FOLDS = 5

LIGHTFM_COMPONENTS = 32
LIGHTFM_EPOCHS = 15
LIGHTFM_LEARNING_RATE = 0.05
LIGHTFM_ITEM_ALPHA = 1e-6
LIGHTFM_USER_ALPHA = 1e-6
LIGHTFM_MAX_SAMPLED = 10
NUM_THREADS = 1

print("Project root:", PROJECT_ROOT)
print("Notebook:", Path.cwd())
print("Dataset:", NAME)
print("Relevance:", f"rating >= {REL}")
print("Validation cutoff:", pd.to_datetime(CUTOFF_TS, unit="s", utc=True))
print("LightFM configuration:", {
    "components": LIGHTFM_COMPONENTS,
    "loss": "warp",
    "epochs": LIGHTFM_EPOCHS,
    "learning_rate": LIGHTFM_LEARNING_RATE,
    "item_alpha": LIGHTFM_ITEM_ALPHA,
    "user_alpha": LIGHTFM_USER_ALPHA,
    "max_sampled": LIGHTFM_MAX_SAMPLED,
    "threads": NUM_THREADS,
})

Project root: /mnt/c/Users/Lenovo/Documents/movie-recommender
Notebook: /mnt/c/Users/Lenovo/Documents/movie-recommender/notebooks
Dataset: sample_10pct
Relevance: rating >= 4.0
Validation cutoff: 2016-01-01 00:00:00+00:00
LightFM configuration: {'components': 32, 'loss': 'warp', 'epochs': 15, 'learning_rate': 0.05, 'item_alpha': 1e-06, 'user_alpha': 1e-06, 'max_sampled': 10, 'threads': 1}


In [2]:
# Load only train/validation and the catalogue + content-v1 artifact.
# Deliberately do not call load_split(), because that helper also reads test.parquet.
processed_dir = PROJECT_ROOT / "data" / "processed" / NAME
train = pd.read_parquet(processed_dir / "train.parquet")
val = pd.read_parquet(processed_dir / "val.parquet")
item_map = pd.read_parquet(processed_dir / "item_map.parquet").sort_values("i").reset_index(drop=True)
user_map = pd.read_parquet(processed_dir / "user_map.parquet").sort_values("u").reset_index(drop=True)

feature_path = PROJECT_ROOT / "data" / "interim" / "hybrid_features" / "content_v1.npz"
if not feature_path.exists():
    raise FileNotFoundError(f"Missing frozen content-v1 feature matrix: {feature_path}")
X_v1 = sp.load_npz(feature_path).tocsr().astype(np.float32)

n_users = int(user_map["u"].max()) + 1
n_items = int(item_map["i"].max()) + 1

assert item_map["i"].is_unique and item_map["movieId"].is_unique
assert user_map["u"].is_unique and user_map["userId"].is_unique
assert np.array_equal(item_map["i"].to_numpy(), np.arange(n_items))
assert np.array_equal(user_map["u"].to_numpy(), np.arange(n_users))
assert X_v1.shape[0] == n_items
assert np.isfinite(X_v1.data).all()
assert {"u", "i", "rating", "timestamp"}.issubset(train.columns)
assert {"u", "i", "rating", "timestamp"}.issubset(val.columns)

print("train:", train.shape, "users:", train["u"].nunique(), "items:", train["i"].nunique())
print("val:  ", val.shape, "users:", val["u"].nunique(), "items:", val["i"].nunique())
print("catalogue:", n_items, "| users:", n_users)
print("content-v1:", X_v1.shape, "nnz:", X_v1.nnz)
print("Test split loaded?", "test" in globals())

# Guardrails: all indices must map into the fixed catalogue/user universe.
for label, frame in [("train", train), ("val", val)]:
    assert frame["u"].between(0, n_users - 1).all(), f"{label}: user index out of bounds"
    assert frame["i"].between(0, n_items - 1).all(), f"{label}: item index out of bounds"

print("Data and mapping checks passed.")

train: (2919680, 4) users: 20093 items: 17236
val:   (137197, 4) users: 20093 items: 10192
catalogue: 17241 | users: 20093
content-v1: (17241, 31) nnz: 53959
Test split loaded? False
Data and mapping checks passed.


## 2 · Build support histories and post-2016 validation queries

A pseudo-new user is a held-out MovieLens user. Their historical positive interactions (`train`, rating ≥ 4) are available only as a pool from which the simulated visitor profile is sampled. Their later validation positives are query targets.

The global popularity/tail labels built here are **only preliminary eligibility estimates**. During the actual five-fold experiment, they must be recomputed separately inside each fold from the non-held-out users only.

In [3]:
# Historical support pool: positive training interactions, one user-item pair once.
train_positive = (
    train.loc[train["rating"] >= REL, ["u", "i"]]
    .drop_duplicates(["u", "i"])
    .astype({"u": "int64", "i": "int64"})
)

support_by_user = (
    train_positive.groupby("u", sort=False)["i"]
    .apply(lambda values: np.asarray(values, dtype=np.int32))
    .to_dict()
)

# Query targets match Notebook 10's positive, post-2016 validation rule.
val_ts = pd.to_numeric(val["timestamp"], errors="raise")
val_window = (
    val.loc[
        (val["rating"] >= REL) & (val_ts >= CUTOFF_TS),
        ["u", "i", "timestamp"],
    ]
    .drop_duplicates(["u", "i"])
    .copy()
)

relevant_by_user = {
    int(user): set(group["i"].astype(int).tolist())
    for user, group in val_window.groupby("u", sort=False)
}

# Confirm train/validation user-item pairs do not overlap.
train_pair_index = pd.MultiIndex.from_frame(train[["u", "i"]].drop_duplicates())
val_pair_index = pd.MultiIndex.from_frame(val_window[["u", "i"]].drop_duplicates())
overlap = len(train_pair_index.intersection(val_pair_index))
assert overlap == 0, f"Found {overlap} train/validation user-item pair overlaps"

print("Positive historical user-item pairs:", f"{len(train_positive):,}")
print("Users with at least one historical positive:", f"{len(support_by_user):,}")
print("Post-2016 positive validation pairs:", f"{len(val_window):,}")
print("Users with at least one validation target:", f"{len(relevant_by_user):,}")
print("Train/validation user-item overlaps:", overlap)

Positive historical user-item pairs: 1,445,945
Users with at least one historical positive: 20,059
Post-2016 positive validation pairs: 25,406
Users with at least one validation target: 5,661
Train/validation user-item overlaps: 0


## 3 · Eligibility and approximate power audit (run this first)

This cell answers four questions before any fold model is fitted:

1. How many users have enough historical positives for each profile size?
2. How many of those users have at least one post-2016 warm-tail validation target?
3. How many such users would fall in each of five user folds?
4. Is the 10% sample large enough to reserve an independent confirmation cohort?

The preliminary warm-tail definition uses the full `sample_10pct` training split solely to estimate eligibility. **It is not the final fold label.** In every fold, a held-out user's interactions will be removed before recomputing item counts and warm-tail membership.

In [4]:
# Preliminary counts only — final warm-tail labels are recomputed within each fold.
positive_counts_prelim = np.bincount(
    train_positive["i"].to_numpy(dtype=np.int64),
    minlength=n_items,
)
warm_tail_mask_prelim = (positive_counts_prelim >= 1) & (positive_counts_prelim < 20)

# All users with a post-2016 positive validation target form the query-user pool.
query_users = np.asarray(sorted(relevant_by_user), dtype=np.int64)
support_count = np.zeros(n_users, dtype=np.int32)
for user_id, items in support_by_user.items():
    support_count[int(user_id)] = len(items)

warm_tail_relevant_prelim = {
    user_id: {item for item in relevant_by_user[user_id] if warm_tail_mask_prelim[item]}
    for user_id in query_users
}

# Stable five-fold assignment over query users. The final training runner will
# exclude each fold's entire user-ID group from global model fitting.
fold_splitter = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_id_by_user = {}
for fold_id, (_, heldout_positions) in enumerate(fold_splitter.split(query_users)):
    for user_id in query_users[heldout_positions]:
        fold_id_by_user[int(user_id)] = int(fold_id)

rows = []
for profile_size in PROFILE_SIZES:
    eligible = [
        int(u) for u in query_users
        if support_count[int(u)] >= profile_size
    ]
    warm_eligible = [
        u for u in eligible
        if len(warm_tail_relevant_prelim[u]) > 0
    ]
    warm_pairs = sum(len(warm_tail_relevant_prelim[u]) for u in eligible)
    fold_counts = [
        sum(fold_id_by_user[u] == f for u in eligible)
        for f in range(N_FOLDS)
    ]
    warm_fold_counts = [
        sum(fold_id_by_user[u] == f for u in warm_eligible)
        for f in range(N_FOLDS)
    ]

    # Approximate 95% half-width for a 1% Bernoulli hit rate. This is a planning
    # heuristic, not a power calculation for the paired ranking metrics.
    n_warm = len(warm_eligible)
    half_width = (
        1.96 * np.sqrt(0.01 * 0.99 / n_warm)
        if n_warm > 0 else np.nan
    )
    rows.append({
        "selected_movies": profile_size,
        "eligible_query_users": len(eligible),
        "warm_tail_query_users_prelim": n_warm,
        "warm_tail_relevant_pairs_prelim": warm_pairs,
        "expected_hits_if_rate_1pct": 0.01 * n_warm,
        "approx_95pct_halfwidth_at_1pct": half_width,
        "fold_user_min": min(fold_counts) if fold_counts else 0,
        "fold_user_max": max(fold_counts) if fold_counts else 0,
        "fold_warm_user_min": min(warm_fold_counts) if warm_fold_counts else 0,
        "fold_warm_user_max": max(warm_fold_counts) if warm_fold_counts else 0,
    })

eligibility_table = pd.DataFrame(rows).set_index("selected_movies")
display(eligibility_table.round(4))

print("\nQuery users assigned to folds:", len(fold_id_by_user))
print("Fold sizes:", pd.Series(list(fold_id_by_user.values())).value_counts().sort_index().to_dict())

# Rough sample-size reference: about 1,521 independent users are needed for a
# ±0.5 percentage-point normal-approximation margin around a 1% hit rate.
required_n_for_halfwidth_0_5pp = int(np.ceil((1.96**2 * 0.01 * 0.99) / (0.005**2)))
print("\nApprox. users needed for ±0.5 pp precision around a 1% rate:", required_n_for_halfwidth_0_5pp)
print("This is only a rough planning reference; paired-difference power depends on discordant users and baseline hit rate.")

,eligible_query_users,warm_tail_query_users_prelim,warm_tail_relevant_pairs_prelim,expected_hits_if_rate_1pct,approx_95pct_halfwidth_at_1pct,fold_user_min,fold_user_max,fold_warm_user_min,fold_warm_user_max
selected_movies,,,,,,,,,
3,5659,931,1533,9.31,0.0064,1131,1133,162,216
5,5653,931,1533,9.31,0.0064,1129,1131,162,216
10,5568,926,1527,9.26,0.0064,1108,1119,161,213
20,4758,885,1473,8.85,0.0066,942,966,153,198



Query users assigned to folds: 5661
Fold sizes: {0: 1133, 1: 1132, 2: 1132, 3: 1132, 4: 1132}

Approx. users needed for ±0.5 pp precision around a 1% rate: 1522
This is only a rough planning reference; paired-difference power depends on discordant users and baseline hit rate.


## 4 · Decision before model fitting

**Do not fit the five fold models yet.** Inspect the eligibility table first.

The next implementation stage will:

1. Assign development and confirmation pseudo-users without mixing user IDs between the groups.
2. Fit fold-specific ALS, LightFM Identity and LightFM Content-v1 models using only non-held-out users.
3. Recompute fold-specific positive counts, warm-tail masks and popularity using only those model-training users.
4. Create repeated support profiles with uniform and popularity-biased sampling.
5. Compare ALS fold-in, ALS-only gate, LightFM centroid, LightFM ridge-style fold-in, and content-to-ALS factor imputation.
6. Use development users to choose only the regularization and rank-percentile / injection-rate policy; freeze those choices before evaluating confirmation users.

### Go/no-go checks to apply after the eligibility audit

- Every profile size being compared must have a clearly reported eligible-user count.
- Report warm-tail eligible users and query pairs separately from overall eligible users.
- If the confirmation cohort has too few warm-tail users to produce informative intervals, do not claim the 10% sample has confirmed the effect. Use an independent cohort from the remaining MovieLens users or treat the result as exploratory.
- ALS-only is an acceptable final outcome. Do not presuppose that LightFM or content-imputed ALS will win.

### Reuse Notebook 10

Notebook 11 reuses the same:

- project root and `sample_10pct` mappings;
- relevance threshold and post-2016 validation window;
- content-v1 feature artifact (`X_v1`);
- implicit ALS half-step mathematics (to be carried over into the fold trainer);
- LightFM WARP hyperparameters and identity-plus-content feature construction;
- top-10 ranking metrics and paired user-bootstrap conventions.

It intentionally does **not** reuse Notebook 10's fitted model outputs as fold models: each fold must refit on the non-held-out users, otherwise the pseudo-new-user experiment would leak their interactions into the global recommender.

## 4 · Development/confirmation cohorts and fold protocol

The first eligibility audit shows that the 10% sample is useful for **development**, but its warm-tail query population is too small for a strong independent confirmation claim. We therefore reserve a small in-sample pilot-confirmation cohort only to test the workflow; it is not the final confirmation result. A stronger confirmation will use disjoint users from the remaining MovieLens users after we verify the full-data user and item mappings.

### User-level split

- Eligible query users have at least 3 historical positive movies and at least one post-2016 positive validation target.
- Reserve 20% of these query users as `pilot_confirmation_users`.
- Split the remaining 80% into five user-grouped development folds.
- For each development fold, exclude **both** that fold's held-out users and all pilot-confirmation users from global model fitting.
- The held-out users' historical `train` rows are used only to simulate the visitor's selected-movie profile. Their rows must not enter the ALS or LightFM fitting interactions.
- Pilot-confirmation users are excluded from every development-fold model. If we later run the in-sample pilot-confirmation check, its model is fitted without those users.

This is deliberately not a nested hyperparameter search. It creates a leakage-safe development structure first. The 10% pilot-confirmation cohort will not be described as adequately powered if the warm-tail sample remains small.

### Fold-specific popularity and warm-tail labels

For each fold, item counts are recomputed from positive interactions of the non-held-out model-training users only:

- strict-cold: 0 positive interactions;
- warm-tail: 1–19 positive interactions;
- head: 20 or more positive interactions.

This prevents a pseudo-new user's own interactions from making their future target appear warm-tail. Popularity-biased support sampling also uses counts from the fold's model-training users, never the held-out user's own history.

In [5]:
# Deterministic development / pilot-confirmation split.
# This only assigns user IDs; it does not fit a recommender.
MIN_PROFILE_SIZE_FOR_COHORT = min(PROFILE_SIZES)
PILOT_CONFIRMATION_FRACTION = 0.20

eligible_query_users_min3 = np.asarray(
    [int(u) for u in query_users if support_count[int(u)] >= MIN_PROFILE_SIZE_FOR_COHORT],
    dtype=np.int64,
)
assert len(eligible_query_users_min3) > N_FOLDS

_split_rng = np.random.default_rng(SEED + 110)
_shuffled_users = _split_rng.permutation(eligible_query_users_min3)
n_pilot_confirm = max(1, int(round(len(_shuffled_users) * PILOT_CONFIRMATION_FRACTION)))
pilot_confirmation_users = np.sort(_shuffled_users[:n_pilot_confirm])
development_users = np.sort(_shuffled_users[n_pilot_confirm:])

_dev_kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED + 111)
development_fold_users = {}
development_fold_id_by_user = {}
for _fold_id, (_, _heldout_positions) in enumerate(_dev_kfold.split(development_users)):
    _heldout = np.sort(development_users[_heldout_positions])
    development_fold_users[int(_fold_id)] = _heldout
    for _uid in _heldout:
        development_fold_id_by_user[int(_uid)] = int(_fold_id)

assert len(set(development_users) & set(pilot_confirmation_users)) == 0
assert len(development_fold_id_by_user) == len(development_users)
assert set(development_fold_id_by_user) == set(map(int, development_users))
assert set(development_users) | set(pilot_confirmation_users) == set(map(int, eligible_query_users_min3))

_split_rows = []
for _split_name, _users in [
    ("development", development_users),
    ("pilot_confirmation", pilot_confirmation_users),
]:
    _split_rows.append({
        "cohort": _split_name,
        "query_users": len(_users),
        "eligible_profile_3": int(np.sum(support_count[_users] >= 3)),
        "eligible_profile_5": int(np.sum(support_count[_users] >= 5)),
        "eligible_profile_10": int(np.sum(support_count[_users] >= 10)),
        "eligible_profile_20": int(np.sum(support_count[_users] >= 20)),
        "overall_validation_pairs": int(sum(len(relevant_by_user[int(u)]) for u in _users)),
    })
cohort_summary = pd.DataFrame(_split_rows).set_index("cohort")
print("Eligible query users (minimum 3 historical positives):", len(eligible_query_users_min3))
print("Development folds:", {k: len(v) for k, v in development_fold_users.items()})
display(cohort_summary)

_fold_summary_rows = []
for _fold_id, _users in development_fold_users.items():
    _fold_summary_rows.append({
        "development_fold": _fold_id,
        "heldout_users": len(_users),
        **{
            f"eligible_profile_{_p}": int(np.sum(support_count[_users] >= _p))
            for _p in PROFILE_SIZES
        },
        "overall_validation_pairs": int(sum(len(relevant_by_user[int(u)]) for u in _users)),
    })
fold_eligibility_summary = pd.DataFrame(_fold_summary_rows).set_index("development_fold")
display(fold_eligibility_summary)

print(
    "Reminder: pilot_confirmation_users are reserved from development fitting, "
    "but this in-sample cohort is not the final independent confirmation."
)

Eligible query users (minimum 3 historical positives): 5659
Development folds: {0: 906, 1: 906, 2: 905, 3: 905, 4: 905}


,query_users,eligible_profile_3,eligible_profile_5,eligible_profile_10,eligible_profile_20,overall_validation_pairs
cohort,,,,,,
development,4527,4527,4523,4455,3788,20284
pilot_confirmation,1132,1132,1130,1113,970,5119


,heldout_users,eligible_profile_3,eligible_profile_5,eligible_profile_10,eligible_profile_20,overall_validation_pairs
development_fold,,,,,,
0,906,906,906,893,746,3964
1,906,906,906,895,746,4086
2,905,905,902,891,771,4120
3,905,905,905,886,756,4030
4,905,905,904,890,769,4084


Reminder: pilot_confirmation_users are reserved from development fitting, but this in-sample cohort is not the final independent confirmation.


## 5 · Reuse Notebook 10's exact LightFM item-feature construction

Notebook 10's `content-v1` LightFM model did not consume the 31-column metadata matrix alone. It used an **identity + metadata** item-feature matrix: explicit item identity features plus the normalized metadata block, with a 0.5 metadata mass for rows with metadata. The identity-only control used the identity matrix alone.

Recreate those matrices exactly here. The matrices depend only on catalogue metadata, not on held-out users or their validation targets, so they can be shared across folds. We will fit new LightFM models inside each fold; we will not reuse Notebook 10's fitted LightFM user/item embeddings.

In [6]:
from sklearn.preprocessing import normalize

assert X_v1.shape == (n_items, 31), (
    f"Expected Notebook 10 content-v1 matrix with 31 columns, got {X_v1.shape}"
)

identity_item_features = sp.eye(n_items, format="csr", dtype=np.float32)

def make_identity_plus_metadata_features(X_side, metadata_mass=0.5):
    """Notebook 10 implementation: identity block + L1-normalized metadata block."""
    X_side = X_side.tocsr().astype(np.float32)
    if X_side.shape[0] != n_items:
        raise ValueError(f"Expected {n_items} item rows, got {X_side.shape[0]}")
    if not (0.0 <= metadata_mass <= 1.0):
        raise ValueError("metadata_mass must be between 0 and 1")

    X_side = normalize(
        X_side, norm="l1", axis=1, copy=True
    ).tocsr().astype(np.float32)
    has_metadata = np.asarray(X_side.getnnz(axis=1) > 0).ravel()
    identity_weights = np.where(
        has_metadata, 1.0 - metadata_mass, 1.0
    ).astype(np.float32)
    identity_block = sp.diags(identity_weights, format="csr") @ identity_item_features
    metadata_block = X_side.multiply(metadata_mass).tocsr()
    features = sp.hstack(
        [identity_block, metadata_block], format="csr", dtype=np.float32
    )
    row_sums = np.asarray(features.sum(axis=1)).ravel()
    if not np.allclose(row_sums, 1.0, atol=1e-5):
        raise AssertionError(
            f"Item feature row weights must sum to one; range={row_sums.min():.6f}..{row_sums.max():.6f}"
        )
    return features

content_v1_item_features = make_identity_plus_metadata_features(
    X_v1, metadata_mass=0.5
)

assert identity_item_features.shape == (n_items, n_items)
assert content_v1_item_features.shape == (n_items, n_items + X_v1.shape[1])
assert np.isfinite(content_v1_item_features.data).all()

print("Identity features:", identity_item_features.shape, "nnz:", identity_item_features.nnz)
print("Content-v1 identity+metadata features:", content_v1_item_features.shape,
      "nnz:", content_v1_item_features.nnz)
print("Feature construction matches Notebook 10's implementation.")

Identity features: (17241, 17241) nnz: 17241
Content-v1 identity+metadata features: (17241, 17272) nnz: 71200
Feature construction matches Notebook 10's implementation.


## 6 · Build one leakage-safe fold context (data-preparation smoke test)

This section prepares one development fold but **does not train ALS or LightFM yet**. It checks the critical data boundaries before we spend time on model fitting.

For the selected smoke-test fold:

1. Exclude the held-out development users and all pilot-confirmation users from `train`.
2. Recompute positive item counts, warm-tail labels, strict-cold labels and all-rating popularity from the remaining users only.
3. Build the fold's positive interaction matrix for LightFM.
4. Retain the selected users' historical positive movies only as profile-support pools.
5. Sample the same support profiles for all methods using uniform and popularity-biased schemes.
6. Report how much of each support profile is represented by the fold-training collaborative history. This is important because ALS and identity-only LightFM cannot learn collaborative factors for items unseen by their fold-training users; content features may still provide a signal.
7. Build overall, warm-tail and strict-cold validation target dictionaries using the fold-specific item labels.

Primary evaluation will exclude only the selected support movies. Full-history exclusion will be a separate sensitivity analysis later.

In [7]:
SMOKE_FOLD_ID = 0
SMOKE_SUPPORT_REPEATS = 2
SUPPORT_SAMPLING_MODES = ("uniform", "popularity_biased")


def _build_positive_interactions(frame):
    """Binary LightFM interactions from unique rating>=REL user-item pairs."""
    pos = (
        frame.loc[frame["rating"] >= REL, ["u", "i"]]
        .drop_duplicates(["u", "i"])
        .astype({"u": "int32", "i": "int32"})
    )
    mat = sp.coo_matrix(
        (
            np.ones(len(pos), dtype=np.float32),
            (pos["u"].to_numpy(), pos["i"].to_numpy()),
        ),
        shape=(n_users, n_items),
        dtype=np.float32,
    )
    mat.sum_duplicates()
    mat.eliminate_zeros()
    if mat.nnz != len(pos):
        raise AssertionError("Positive interaction matrix contains duplicate pairs")
    return pos, mat


def _build_fold_context(fold_id, support_repeats=2):
    if int(fold_id) not in development_fold_users:
        raise KeyError(f"Unknown development fold: {fold_id}")
    if support_repeats < 1:
        raise ValueError("support_repeats must be >= 1")

    heldout_users = np.asarray(development_fold_users[int(fold_id)], dtype=np.int64)
    # Confirmation users are held out from every development model fit.
    excluded_users = np.union1d(heldout_users, pilot_confirmation_users).astype(np.int64)
    excluded_set = set(map(int, excluded_users))
    fold_train = train.loc[~train["u"].isin(excluded_set)].copy()

    # Strong leakage assertion: no row from either held-out cohort enters model fitting.
    if fold_train["u"].isin(excluded_set).any():
        raise AssertionError("Held-out user rows leaked into fold_train")

    fold_positive, interactions_fold = _build_positive_interactions(fold_train)
    positive_counts_fold = np.bincount(
        fold_positive["i"].to_numpy(dtype=np.int64), minlength=n_items
    )
    # App-style popularity is based on all ratings from model-training users.
    popularity_counts_fold = np.bincount(
        fold_train["i"].to_numpy(dtype=np.int64), minlength=n_items
    ).astype(np.float64)

    warm_tail_mask_fold = (positive_counts_fold >= 1) & (positive_counts_fold < 20)
    strict_cold_mask_fold = positive_counts_fold == 0
    head_mask_fold = positive_counts_fold >= 20
    item_has_positive_history_fold = positive_counts_fold > 0

    # Held-out users' validation targets, labelled only by fold-training popularity.
    fold_overall_relevant = {
        int(u): set(relevant_by_user[int(u)])
        for u in heldout_users if int(u) in relevant_by_user
    }
    fold_warm_tail_relevant = {
        u: {i for i in items if warm_tail_mask_fold[i]}
        for u, items in fold_overall_relevant.items()
    }
    fold_strict_cold_relevant = {
        u: {i for i in items if strict_cold_mask_fold[i]}
        for u, items in fold_overall_relevant.items()
    }
    fold_warm_tail_relevant = {
        u: items for u, items in fold_warm_tail_relevant.items() if items
    }
    fold_strict_cold_relevant = {
        u: items for u, items in fold_strict_cold_relevant.items() if items
    }

    # No validation target for a held-out user should occur in that user's train support.
    for user_id, target_items in fold_overall_relevant.items():
        support_items = set(map(int, support_by_user.get(user_id, np.array([], dtype=np.int32))))
        if support_items & target_items:
            raise AssertionError(
                f"Train/validation target overlap for held-out user {user_id}"
            )

    # Confirm LightFM fitting interactions contain no held-out rows.
    interactions_fold_csr = interactions_fold.tocsr()
    if interactions_fold_csr[excluded_users].nnz != 0:
        raise AssertionError("Held-out users have nonzero rows in fold LightFM interactions")

    # Generate deterministic support samples. The same stored profiles are reused
    # across all candidate methods so differences are not caused by different picks.
    support_samples = {}
    audit_rows = []
    for user_id in heldout_users:
        user_id = int(user_id)
        pool = np.asarray(support_by_user.get(user_id, np.array([], dtype=np.int32)), dtype=np.int32)
        if len(pool) == 0:
            continue
        user_targets = fold_overall_relevant.get(user_id, set())
        warm_target_count = len(fold_warm_tail_relevant.get(user_id, set()))
        cold_target_count = len(fold_strict_cold_relevant.get(user_id, set()))

        for profile_size in PROFILE_SIZES:
            if len(pool) < profile_size:
                continue
            for mode_id, mode in enumerate(SUPPORT_SAMPLING_MODES):
                for repeat_id in range(support_repeats):
                    rng = np.random.default_rng(
                        np.random.SeedSequence([
                            SEED, int(fold_id), user_id, int(profile_size),
                            int(mode_id), int(repeat_id),
                        ])
                    )
                    if mode == "uniform":
                        chosen = rng.choice(pool, size=profile_size, replace=False)
                    elif mode == "popularity_biased":
                        # Sensitivity sampler: popular historical positives are more
                        # likely to be selected, while every item retains nonzero chance.
                        weights = np.sqrt(np.maximum(popularity_counts_fold[pool], 1.0))
                        weights = weights / weights.sum()
                        chosen = rng.choice(
                            pool, size=profile_size, replace=False, p=weights
                        )
                    else:
                        raise ValueError(f"Unknown support sampling mode: {mode}")

                    chosen = np.asarray(chosen, dtype=np.int32)
                    if len(np.unique(chosen)) != profile_size:
                        raise AssertionError("Support profile contains duplicate movies")
                    if set(map(int, chosen)) & user_targets:
                        raise AssertionError("A validation target entered the support profile")

                    covered = item_has_positive_history_fold[chosen]
                    key = (user_id, int(profile_size), mode, int(repeat_id))
                    support_samples[key] = chosen
                    audit_rows.append({
                        "user_id": user_id,
                        "profile_size": int(profile_size),
                        "sampling_mode": mode,
                        "repeat_id": int(repeat_id),
                        "selected_movies": len(chosen),
                        "support_items_with_fold_collaborative_history": int(covered.sum()),
                        "support_item_coverage_fraction": float(covered.mean()),
                        "overall_validation_targets": len(user_targets),
                        "warm_tail_validation_targets": warm_target_count,
                        "strict_cold_validation_targets": cold_target_count,
                    })

    support_audit = pd.DataFrame(audit_rows)

    # Warm-tail and strict-cold pair counts are unique user-item pairs, not sample repeats.
    summary = {
        "fold_id": int(fold_id),
        "heldout_development_users": len(heldout_users),
        "pilot_confirmation_users_excluded": len(pilot_confirmation_users),
        "model_training_users_with_any_rows": int(fold_train["u"].nunique()),
        "model_training_rating_rows": int(len(fold_train)),
        "model_training_positive_pairs": int(len(fold_positive)),
        "catalogue_items_with_positive_history": int(item_has_positive_history_fold.sum()),
        "warm_tail_catalogue_items": int(warm_tail_mask_fold.sum()),
        "strict_cold_catalogue_items": int(strict_cold_mask_fold.sum()),
        "head_catalogue_items": int(head_mask_fold.sum()),
        "heldout_users_with_overall_targets": len(fold_overall_relevant),
        "heldout_users_with_warm_tail_targets": len(fold_warm_tail_relevant),
        "warm_tail_validation_pairs": int(sum(map(len, fold_warm_tail_relevant.values()))),
        "strict_cold_validation_pairs": int(sum(map(len, fold_strict_cold_relevant.values()))),
    }

    return {
        "summary": summary,
        "fold_train": fold_train,
        "fold_positive": fold_positive,
        "interactions": interactions_fold,
        "positive_counts": positive_counts_fold,
        "popularity_counts": popularity_counts_fold,
        "warm_tail_mask": warm_tail_mask_fold,
        "strict_cold_mask": strict_cold_mask_fold,
        "head_mask": head_mask_fold,
        "item_has_positive_history": item_has_positive_history_fold,
        "heldout_users": heldout_users,
        "excluded_users": excluded_users,
        "overall_relevant": fold_overall_relevant,
        "warm_tail_relevant": fold_warm_tail_relevant,
        "strict_cold_relevant": fold_strict_cold_relevant,
        "support_samples": support_samples,
        "support_audit": support_audit,
    }


# One-fold data-preparation smoke test. No ALS/LightFM fitting occurs here.
smoke_fold = _build_fold_context(
    SMOKE_FOLD_ID, support_repeats=SMOKE_SUPPORT_REPEATS
)

print("=== Fold training and leakage audit ===")
display(pd.Series(smoke_fold["summary"], name="value").to_frame())

print("=== Support sampling audit ===")
smoke_support_summary = (
    smoke_fold["support_audit"]
    .groupby(["profile_size", "sampling_mode"])
    .agg(
        eligible_users=("user_id", "nunique"),
        support_samples=("selected_movies", "size"),
        mean_collaborative_support_coverage=("support_item_coverage_fraction", "mean"),
        p10_collaborative_support_coverage=("support_item_coverage_fraction", lambda x: x.quantile(0.10)),
        warm_tail_query_users=("warm_tail_validation_targets", lambda x: int((x > 0).groupby(smoke_fold["support_audit"].loc[x.index, "user_id"]).any().sum())),
    )
)
display(smoke_support_summary.round(4))

# Final boundary assertions for the smoke fold.
assert smoke_fold["interactions"].shape == (n_users, n_items)
assert smoke_fold["interactions"].nnz == len(smoke_fold["fold_positive"])
assert not smoke_fold["fold_train"]["u"].isin(smoke_fold["excluded_users"]).any()
assert smoke_fold["interactions"].tocsr()[smoke_fold["excluded_users"]].nnz == 0
assert smoke_fold["warm_tail_mask"].sum() + smoke_fold["strict_cold_mask"].sum() + smoke_fold["head_mask"].sum() == n_items
assert all(
    set(items).issubset(set(smoke_fold["overall_relevant"][u]))
    for u, items in smoke_fold["warm_tail_relevant"].items()
)
print("\nPASS: fold-specific data preparation, user exclusion, support sampling and target-scope checks.")
print("No model fitting has been run. No test split has been loaded.")

=== Fold training and leakage audit ===


,value
fold_id,0
heldout_development_users,906
pilot_confirmation_users_excluded,1132
model_training_users_with_any_rows,18055
model_training_rating_rows,2533620
model_training_positive_pairs,1248910
catalogue_items_with_positive_history,15639
warm_tail_catalogue_items,10469
strict_cold_catalogue_items,1602
head_catalogue_items,5170


=== Support sampling audit ===


eligible_users  support_samples  \
profile_size sampling_mode                                        
3            popularity_biased             906             1812   
             uniform                       906             1812   
5            popularity_biased             906             1812   
             uniform                       906             1812   
10           popularity_biased             893             1786   
             uniform                       893             1786   
20           popularity_biased             746             1492   
             uniform                       746             1492   

                                mean_collaborative_support_coverage  \
profile_size sampling_mode                                            
3            popularity_biased                               0.9994   
             uniform                                         0.9980   
5            popularity_biased                               1.0000   
             uniform                                         0.9978   
10           popularity_biased                               0.9998   
             uniform                                         0.9983   
20           popularity_biased                               0.9998   
             uniform                                         0.9986   

                                p10_collaborative_support_coverage  \
profile_size sampling_mode                                           
3            popularity_biased                                 1.0   
             uniform                                           1.0   
5            popularity_biased                                 1.0   
             uniform                                           1.0   
10           popularity_biased                                 1.0   
             uniform                                           1.0   
20           popularity_biased                                 1.0   
             uniform                                           1.0   

                                warm_tail_query_users  
profile_size sampling_mode                             
3            popularity_biased                    178  
             uniform                              178  
5            popularity_biased                    178  
             uniform                              178  
10           popularity_biased                    178  
             uniform                              178  
20           popularity_biased                    164  
             uniform                              164


PASS: fold-specific data preparation, user exclusion, support sampling and target-scope checks.
No model fitting has been run. No test split has been loaded.


## 7 · Stop point: review the fold-preparation output before fitting

Do **not** run five-fold model training yet. First inspect:

- the number of held-out development users and warm-tail target users in the smoke fold;
- the warm-tail / strict-cold catalogue sizes recomputed without the held-out users;
- support-item collaborative coverage at each profile size;
- the difference between uniform and popularity-biased support profiles.

The support coverage statistic is important. It tells us how many selected movies have collaborative interactions among the fold-training users. A held-out user's selected movie can be absent from the fold model's collaborative history even though it is present in the catalogue. The identity-only and ALS arms will have limited signal for such items, while content features and content-to-ALS imputation may help.

### What this phase does not do

- It does not fit ALS or LightFM models.
- It does not load or inspect `test.parquet`.
- It does not select a winning method or tune a gate.
- It does not claim that the in-sample pilot-confirmation cohort is adequately powered.

### When we use the remaining 90%

We do not need full-data training for this fold-preparation step or the initial development smoke test. Once the development methods and settings are stable, we should use users outside `sample_10pct` for an independent confirmation design. At that point we will need to read the original full ratings file (or a disjoint user-level extract), verify catalogue coverage/mapping for those users, and decide whether to fit the global model on the 10% development population or a larger user population. I will flag that transition before any full-data fit.

**Next after reviewing this output:** fit one smoke fold only (ALS + LightFM Identity + LightFM Content-v1), verify inference/scoring parity and runtime, then decide whether to launch the full five-fold development run.

In [8]:
import time
import numpy as np
import scipy.sparse as sp

ALS_FACTORS = 64
ALS_ALPHA = 5.0
ALS_LAMBDA = 80.0
ALS_ITERATIONS = 10


def _implicit_half_step(M, Y, lam):
    """One implicit ALS half-step."""
    n_rows = M.shape[0]
    k = Y.shape[1]

    YtY = Y.T @ Y
    reg = lam * np.eye(k, dtype=np.float64)

    out = np.zeros((n_rows, k), dtype=np.float64)
    indptr, indices, data = M.indptr, M.indices, M.data

    for row in range(n_rows):
        start, end = indptr[row], indptr[row + 1]

        if start == end:
            continue

        factors = Y[indices[start:end]]
        confidence_minus_one = data[start:end]

        A = (
            YtY
            + (factors.T * confidence_minus_one) @ factors
            + reg
        )

        b = factors.T @ (confidence_minus_one + 1.0)
        out[row] = np.linalg.solve(A, b)

    return out


def fit_fold_implicit_als(
    df,
    k=ALS_FACTORS,
    alpha=ALS_ALPHA,
    lam=ALS_LAMBDA,
    n_iter=ALS_ITERATIONS,
    seed=SEED,
):
    positives = (
        df.loc[df["rating"] >= REL, ["u", "i"]]
        .drop_duplicates(["u", "i"])
    )

    C = sp.csr_matrix(
        (
            np.full(len(positives), float(alpha), dtype=np.float64),
            (
                positives["u"].to_numpy(dtype=np.int64),
                positives["i"].to_numpy(dtype=np.int64),
            ),
        ),
        shape=(n_users, n_items),
    )

    C.sum_duplicates()
    C.eliminate_zeros()

    assert C.nnz == len(positives)

    Ct = C.T.tocsr()

    rng = np.random.default_rng(seed)
    Q = rng.normal(0, 0.1, size=(n_items, k))
    P = np.zeros((n_users, k), dtype=np.float64)

    start = time.perf_counter()

    for iteration in range(1, n_iter + 1):
        P = _implicit_half_step(C, Q, lam)
        Q = _implicit_half_step(Ct, P, lam)

        if iteration == 1 or iteration % 2 == 0 or iteration == n_iter:
            print(
                f"ALS iteration {iteration:02d}/{n_iter} | "
                f"{time.perf_counter() - start:.1f}s"
            )

    return {
        "P": P.astype(np.float32),
        "Q": Q.astype(np.float32),
        "positive_interactions": C.nnz,
        "training_time_seconds": time.perf_counter() - start,
    }


print("Fitting ALS on fold-0 training data...")

als_smoke = fit_fold_implicit_als(
    smoke_fold["fold_train"]
)

P_als_smoke = als_smoke["P"]
Q_als_smoke = als_smoke["Q"]

item_has_als_factors_smoke = (
    np.abs(Q_als_smoke).sum(axis=1) > 0
)

assert P_als_smoke.shape == (n_users, ALS_FACTORS)
assert Q_als_smoke.shape == (n_items, ALS_FACTORS)

assert np.isfinite(P_als_smoke).all()
assert np.isfinite(Q_als_smoke).all()

# Strict-cold items must have no learned ALS item factors.
assert np.array_equal(
    ~item_has_als_factors_smoke,
    smoke_fold["strict_cold_mask"],
)

# Confirm no excluded users entered ALS training.
assert not smoke_fold["fold_train"]["u"].isin(
    smoke_fold["excluded_users"]
).any()

print("\nALS smoke test passed.")
print("P shape:", P_als_smoke.shape)
print("Q shape:", Q_als_smoke.shape)
print("Positive interactions:", f"{als_smoke['positive_interactions']:,}")
print("Items with ALS factors:", int(item_has_als_factors_smoke.sum()))
print("Items without ALS factors:", int((~item_has_als_factors_smoke).sum()))
print(
    "Training time:",
    f"{als_smoke['training_time_seconds'] / 60:.2f} minutes"
)

Fitting ALS on fold-0 training data...
ALS iteration 01/10 | 5.9s
ALS iteration 02/10 | 10.2s
ALS iteration 04/10 | 18.5s
ALS iteration 06/10 | 35.5s
ALS iteration 08/10 | 46.9s
ALS iteration 10/10 | 56.4s

ALS smoke test passed.
P shape: (20093, 64)
Q shape: (17241, 64)
Positive interactions: 1,248,910
Items with ALS factors: 15639
Items without ALS factors: 1602
Training time: 0.94 minutes


In [9]:
import time
import numpy as np
from lightfm import LightFM

lightfm_smoke_features = {
    "Identity": identity_item_features,
    "Content-v1": content_v1_item_features,
}

lightfm_smoke_models = {}
lightfm_smoke_representations = {}


def fit_lightfm_smoke(name, item_features):

    model = LightFM(
        no_components=LIGHTFM_COMPONENTS,
        loss="warp",
        learning_rate=LIGHTFM_LEARNING_RATE,
        item_alpha=LIGHTFM_ITEM_ALPHA,
        user_alpha=LIGHTFM_USER_ALPHA,
        max_sampled=LIGHTFM_MAX_SAMPLED,
        random_state=SEED,
    )

    start = time.perf_counter()

    model.fit(
        interactions=smoke_fold["interactions"],
        item_features=item_features,
        epochs=LIGHTFM_EPOCHS,
        num_threads=NUM_THREADS,
        verbose=True,
    )

    elapsed = time.perf_counter() - start

    # Extract effective item representations.
    # These include the learned feature embeddings multiplied by item features.
    item_bias, item_vectors = model.get_item_representations(
        item_features
    )

    user_bias, user_vectors = model.get_user_representations()

    assert item_vectors.shape == (
        n_items,
        LIGHTFM_COMPONENTS,
    )

    assert user_vectors.shape == (
        n_users,
        LIGHTFM_COMPONENTS,
    )

    assert item_bias.shape == (n_items,)
    assert user_bias.shape == (n_users,)

    assert np.isfinite(item_vectors).all()
    assert np.isfinite(item_bias).all()
    assert np.isfinite(user_vectors).all()
    assert np.isfinite(user_bias).all()

    # Verify direct matrix scoring against LightFM's official predict().
    check_user = int(smoke_fold["fold_train"]["u"].iloc[0])

    check_items = np.unique(
        np.array(
            [0, 1, 17, n_items - 1],
            dtype=np.int32,
        )
    )

    direct_scores = (
        user_vectors[check_user] @ item_vectors[check_items].T
        + user_bias[check_user]
        + item_bias[check_items]
    )

    official_scores = model.predict(
        user_ids=np.full(
            len(check_items),
            check_user,
            dtype=np.int32,
        ),
        item_ids=check_items,
        item_features=item_features,
        num_threads=NUM_THREADS,
    )

    assert np.allclose(
        direct_scores,
        official_scores,
        rtol=1e-4,
        atol=1e-5,
    ), f"{name}: direct scorer does not match LightFM predict()"

    return {
        "model": model,
        "item_vectors": item_vectors,
        "item_bias": item_bias,
        "user_vectors": user_vectors,
        "user_bias": user_bias,
        "training_time_seconds": elapsed,
    }


for name, features in lightfm_smoke_features.items():

    print(f"\n{'=' * 55}")
    print(f"Training LightFM {name}")
    print("=" * 55)

    result = fit_lightfm_smoke(name, features)

    lightfm_smoke_models[name] = result["model"]

    lightfm_smoke_representations[name] = {
        "item_vectors": result["item_vectors"],
        "item_bias": result["item_bias"],
        "user_vectors": result["user_vectors"],
        "user_bias": result["user_bias"],
    }

    print(f"\n{name} training complete.")
    print("Item vectors:", result["item_vectors"].shape)
    print("Item biases:", result["item_bias"].shape)
    print("User vectors:", result["user_vectors"].shape)
    print(
        "Training time:",
        f"{result['training_time_seconds'] / 60:.2f} minutes"
    )
    print("Official predict() parity check passed.")

print("\nBoth LightFM models trained successfully.")


Training LightFM Identity
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14

Identity training complete.
Item vectors: (17241, 32)
Item biases: (17241,)
User vectors: (20093, 32)
Training time: 0.41 minutes
Official predict() parity check passed.

Training LightFM Content-v1
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14

Content-v1 training complete.
Item vectors: (17241, 32)
Item biases: (17241,)
User vectors: (20093, 32)
Training time: 0.62 minutes
Official predict() parity check passed.

Both LightFM models trained successfully.


In [10]:
LIGHTFM_RIDGE_LAMBDA_SMOKE = 1.0
SMOKE_PROFILE_SIZE = 5
SMOKE_TOP_K = 10


# --------------------------------------------------
# 1. ALS anonymous-user fold-in
# --------------------------------------------------

def fold_in_als_profile(
    profile_items,
    Q,
    alpha=ALS_ALPHA,
    lam=ALS_LAMBDA,
):
    profile_items = np.asarray(
        profile_items,
        dtype=np.int64,
    )

    if len(profile_items) == 0:
        raise ValueError("Profile cannot be empty.")

    if len(np.unique(profile_items)) != len(profile_items):
        raise ValueError("Profile contains duplicate items.")

    selected_factors = Q[profile_items].astype(np.float64)

    Y = Q.astype(np.float64)

    A = (
        Y.T @ Y
        + alpha * (selected_factors.T @ selected_factors)
        + lam * np.eye(Y.shape[1])
    )

    b = selected_factors.T @ np.full(
        len(profile_items),
        alpha + 1.0,
    )

    user_vector = np.linalg.solve(A, b)

    assert np.isfinite(user_vector).all()

    return user_vector.astype(np.float32)


# --------------------------------------------------
# 2. LightFM anonymous-user inference
# --------------------------------------------------

def lightfm_centroid_fold_in(profile_items, item_vectors):

    profile_items = np.asarray(profile_items, dtype=np.int64)

    user_vector = item_vectors[profile_items].mean(axis=0)

    assert np.isfinite(user_vector).all()

    return user_vector.astype(np.float32)


def lightfm_ridge_fold_in(
    profile_items,
    item_vectors,
    ridge=LIGHTFM_RIDGE_LAMBDA_SMOKE,
):

    profile_items = np.asarray(profile_items, dtype=np.int64)

    selected_vectors = item_vectors[profile_items].astype(
        np.float64
    )

    k = selected_vectors.shape[1]

    A = (
        selected_vectors.T @ selected_vectors
        + ridge * np.eye(k)
    )

    # Each selected movie is treated as a positive example with target 1.
    b = selected_vectors.T @ np.ones(len(profile_items))

    user_vector = np.linalg.solve(A, b)

    assert np.isfinite(user_vector).all()

    return user_vector.astype(np.float32)


# --------------------------------------------------
# 3. Full-catalogue scoring functions
# --------------------------------------------------

def score_als_profile(user_vector):
    scores = Q_als_smoke @ user_vector

    assert scores.shape == (n_items,)
    assert np.isfinite(scores).all()

    return scores


def score_lightfm_profile(user_vector, item_vectors, item_bias):

    scores = (
        item_vectors @ user_vector
        + item_bias
    )

    assert scores.shape == (n_items,)
    assert np.isfinite(scores).all()

    # The anonymous user bias is omitted because it is constant
    # across all candidate movies and cannot change their ranking.
    return scores


# --------------------------------------------------
# 4. Prepare catalogue segments for diagnostics
# --------------------------------------------------

item_segment = np.full(
    n_items,
    "strict-cold",
    dtype=object,
)

item_segment[smoke_fold["warm_tail_mask"]] = "warm-tail"
item_segment[smoke_fold["head_mask"]] = "head"


def get_top_items(scores, profile_items, eligible_mask=None, k=10):

    row = np.asarray(scores, dtype=np.float64).copy()

    # Do not recommend movies already selected by the visitor.
    row[np.asarray(profile_items, dtype=np.int64)] = -np.inf

    if eligible_mask is not None:
        row[~eligible_mask] = -np.inf

    valid = np.flatnonzero(np.isfinite(row))

    if len(valid) == 0:
        return []

    top = valid[np.argsort(row[valid])[-k:][::-1]]

    return [
        {
            "item_index": int(item),
            "score": float(row[item]),
            "segment": item_segment[item],
        }
        for item in top
    ]


# --------------------------------------------------
# 5. Score three held-out users
# --------------------------------------------------

example_users = sorted(
    smoke_fold["warm_tail_relevant"].keys()
)[:3]

assert len(example_users) == 3

smoke_scoring_results = []

for user_id in example_users:

    profile_items = smoke_fold["support_samples"][
        (
            int(user_id),
            SMOKE_PROFILE_SIZE,
            "uniform",
            0,
        )
    ]

    print(f"\n{'=' * 65}")
    print("Held-out user:", user_id)
    print("Profile items:", profile_items.tolist())
    print("Profile size:", len(profile_items))
    print(
        "Warm-tail validation targets:",
        len(smoke_fold["warm_tail_relevant"][user_id]),
    )

    # ALS
    als_user_vector = fold_in_als_profile(
        profile_items,
        Q_als_smoke,
    )

    als_scores = score_als_profile(als_user_vector)

    # LightFM Identity
    identity_reps = lightfm_smoke_representations["Identity"]

    identity_centroid = lightfm_centroid_fold_in(
        profile_items,
        identity_reps["item_vectors"],
    )

    identity_ridge = lightfm_ridge_fold_in(
        profile_items,
        identity_reps["item_vectors"],
    )

    identity_centroid_scores = score_lightfm_profile(
        identity_centroid,
        identity_reps["item_vectors"],
        identity_reps["item_bias"],
    )

    identity_ridge_scores = score_lightfm_profile(
        identity_ridge,
        identity_reps["item_vectors"],
        identity_reps["item_bias"],
    )

    # LightFM Content-v1
    content_reps = lightfm_smoke_representations["Content-v1"]

    content_centroid = lightfm_centroid_fold_in(
        profile_items,
        content_reps["item_vectors"],
    )

    content_ridge = lightfm_ridge_fold_in(
        profile_items,
        content_reps["item_vectors"],
    )

    content_centroid_scores = score_lightfm_profile(
        content_centroid,
        content_reps["item_vectors"],
        content_reps["item_bias"],
    )

    content_ridge_scores = score_lightfm_profile(
        content_ridge,
        content_reps["item_vectors"],
        content_reps["item_bias"],
    )

    methods = {
        "ALS fold-in": als_scores,
        "LightFM Identity centroid": identity_centroid_scores,
        "LightFM Identity ridge": identity_ridge_scores,
        "LightFM Content-v1 centroid": content_centroid_scores,
        "LightFM Content-v1 ridge": content_ridge_scores,
    }

    for method_name, scores in methods.items():

        # Full catalogue: all n_items receive a score.
        assert scores.shape == (n_items,)
        assert np.isfinite(scores).all()

        # Show the unrestricted full-catalogue ranking.
        full_top10 = get_top_items(
            scores,
            profile_items,
            eligible_mask=None,
            k=SMOKE_TOP_K,
        )

        # Also show the ranking restricted to items with fold-training
        # collaborative history. This is the useful warm-tail comparison pool.
        collaborative_top10 = get_top_items(
            scores,
            profile_items,
            eligible_mask=smoke_fold["item_has_positive_history"],
            k=SMOKE_TOP_K,
        )

        smoke_scoring_results.append({
            "user_id": int(user_id),
            "method": method_name,
            "catalogue_scored": len(scores),
            "full_catalogue_top10": [
                x["item_index"] for x in full_top10
            ],
            "full_top10_segments": [
                x["segment"] for x in full_top10
            ],
            "collaborative_top10": [
                x["item_index"] for x in collaborative_top10
            ],
        })

        print(f"\n{method_name}")
        print("Full-catalogue top 10:")
        print(full_top10)
        print("Top 10 with collaborative history:")
        print(collaborative_top10)


smoke_scoring_results = pd.DataFrame(smoke_scoring_results)

print("\n=== Final scoring summary ===")
display(smoke_scoring_results)

print("\nAll profile vectors and full-catalogue scores passed numerical checks.")


Held-out user: 182
Profile items: [1170, 14294, 4396, 7027, 2636]
Profile size: 5
Warm-tail validation targets: 1

ALS fold-in
Full-catalogue top 10:
[{'item_index': 0, 'score': 0.23947443068027496, 'segment': 'head'}, {'item_index': 5650, 'score': 0.2299172282218933, 'segment': 'head'}, {'item_index': 3868, 'score': 0.21674996614456177, 'segment': 'head'}, {'item_index': 1798, 'score': 0.21575824916362762, 'segment': 'head'}, {'item_index': 459, 'score': 0.2018146812915802, 'segment': 'head'}, {'item_index': 2817, 'score': 0.1994885802268982, 'segment': 'head'}, {'item_index': 1140, 'score': 0.19030465185642242, 'segment': 'head'}, {'item_index': 7261, 'score': 0.1853785365819931, 'segment': 'head'}, {'item_index': 1165, 'score': 0.1781143844127655, 'segment': 'head'}, {'item_index': 1435, 'score': 0.17772048711776733, 'segment': 'head'}]
Top 10 with collaborative history:
[{'item_index': 0, 'score': 0.23947443068027496, 'segment': 'head'}, {'item_index': 5650, 'score': 0.22991722822

,user_id,method,catalogue_scored,full_catalogue_top10,full_top10_segments,collaborative_top10
0,182,ALS fold-in,17241,"[0, 5650, 3868, 1798, 459, 2817, 1140, 7261, 1...","[head, head, head, head, head, head, head, hea...","[0, 5650, 3868, 1798, 459, 2817, 1140, 7261, 1..."
1,182,LightFM Identity centroid,17241,"[2320, 1099, 2675, 4493, 249, 1101, 283, 5297,...","[head, head, head, head, head, head, head, hea...","[2320, 1099, 2675, 4493, 249, 1101, 283, 5297,..."
2,182,LightFM Identity ridge,17241,"[283, 2320, 305, 2675, 563, 249, 803, 1099, 44...","[head, head, head, head, head, head, head, hea...","[283, 2320, 305, 2675, 563, 249, 803, 1099, 44..."
3,182,LightFM Content-v1 centroid,17241,"[2320, 4493, 305, 1099, 2675, 5297, 249, 342, ...","[head, head, head, head, head, head, head, hea...","[2320, 4493, 305, 1099, 2675, 5297, 249, 342, ..."
4,182,LightFM Content-v1 ridge,17241,"[2320, 305, 283, 563, 2675, 46, 342, 249, 4493...","[head, head, head, head, head, head, head, hea...","[2320, 305, 283, 563, 2675, 46, 342, 249, 4493..."
5,499,ALS fold-in,17241,"[563, 305, 49, 342, 46, 6062, 576, 6519, 561, ...","[head, head, head, head, head, head, head, hea...","[563, 305, 49, 342, 46, 6062, 576, 6519, 561, ..."
6,499,LightFM Identity centroid,17241,"[305, 342, 10930, 9698, 12614, 6302, 563, 4493...","[head, head, head, head, head, head, head, hea...","[305, 342, 10930, 9698, 12614, 6302, 563, 4493..."
7,499,LightFM Identity ridge,17241,"[305, 342, 563, 506, 46, 10930, 12614, 9698, 0...","[head, head, head, head, head, head, head, hea...","[305, 342, 563, 506, 46, 10930, 12614, 9698, 0..."
8,499,LightFM Content-v1 centroid,17241,"[10930, 305, 342, 12614, 11688, 9698, 12488, 2...","[head, head, head, head, head, head, head, hea...","[10930, 305, 342, 12614, 11688, 9698, 12488, 2..."
9,499,LightFM Content-v1 ridge,17241,"[305, 342, 2675, 10930, 9698, 563, 506, 2320, ...","[head, head, head, head, head, head, head, hea...","[305, 342, 2675, 10930, 9698, 563, 506, 2320, ..."



All profile vectors and full-catalogue scores passed numerical checks.


In [11]:

from collections import defaultdict

EVAL_K = 10
eval_start = time.perf_counter()

# --------------------------------------------------
# 1. Define the five inference methods
# --------------------------------------------------

identity_reps = lightfm_smoke_representations["Identity"]
content_reps = lightfm_smoke_representations["Content-v1"]

def score_all_methods(profile_items):

    # ALS anonymous-user fold-in
    als_vector = fold_in_als_profile(
        profile_items,
        Q_als_smoke,
    )
    als_scores = Q_als_smoke @ als_vector

    # LightFM Identity
    identity_centroid = lightfm_centroid_fold_in(
        profile_items,
        identity_reps["item_vectors"],
    )
    identity_ridge = lightfm_ridge_fold_in(
        profile_items,
        identity_reps["item_vectors"],
    )

    identity_centroid_scores = score_lightfm_profile(
        identity_centroid,
        identity_reps["item_vectors"],
        identity_reps["item_bias"],
    )
    identity_ridge_scores = score_lightfm_profile(
        identity_ridge,
        identity_reps["item_vectors"],
        identity_reps["item_bias"],
    )

    # LightFM Content-v1
    content_centroid = lightfm_centroid_fold_in(
        profile_items,
        content_reps["item_vectors"],
    )
    content_ridge = lightfm_ridge_fold_in(
        profile_items,
        content_reps["item_vectors"],
    )

    content_centroid_scores = score_lightfm_profile(
        content_centroid,
        content_reps["item_vectors"],
        content_reps["item_bias"],
    )
    content_ridge_scores = score_lightfm_profile(
        content_ridge,
        content_reps["item_vectors"],
        content_reps["item_bias"],
    )

    return {
        "ALS fold-in": als_scores,
        "LightFM Identity centroid": identity_centroid_scores,
        "LightFM Identity ridge": identity_ridge_scores,
        "LightFM Content-v1 centroid": content_centroid_scores,
        "LightFM Content-v1 ridge": content_ridge_scores,
    }


# --------------------------------------------------
# 2. Deterministic top-K selection
# --------------------------------------------------

def stable_top_k(scores, k=EVAL_K):

    valid = np.flatnonzero(np.isfinite(scores))

    if len(valid) == 0:
        return np.array([], dtype=np.int64)

    k = min(k, len(valid))
    valid_scores = scores[valid]

    # Find the score at the top-K boundary without sorting the
    # entire catalogue.
    threshold = np.partition(
        valid_scores,
        len(valid) - k,
    )[len(valid) - k]

    above = valid[scores[valid] > threshold]
    tied = valid[scores[valid] == threshold]

    remaining = k - len(above)
    chosen = np.concatenate([above, tied[:remaining]])

    # Deterministic tie-break: lower item index comes first.
    order = np.lexsort((chosen, -scores[chosen]))

    return chosen[order]


# --------------------------------------------------
# 3. Ranking metrics
# --------------------------------------------------

def calculate_ranking_metrics(top_items, relevant_items, k=EVAL_K):

    relevant_items = set(relevant_items)

    if not relevant_items:
        return {
            "ndcg": np.nan,
            "recall": np.nan,
            "hit": np.nan,
            "hits": 0,
        }

    hits = [
        rank
        for rank, item in enumerate(top_items, start=1)
        if int(item) in relevant_items
    ]

    dcg = sum(
        1.0 / np.log2(rank + 1)
        for rank in hits
    )

    ideal_hits = min(len(relevant_items), k)

    idcg = sum(
        1.0 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    return {
        "ndcg": dcg / idcg if idcg else 0.0,
        "recall": len(hits) / len(relevant_items),
        "hit": float(len(hits) > 0),
        "hits": len(hits),
    }


def get_full_catalogue_rank(scores, item_id):

    target_score = scores[item_id]

    # Count all items ranked ahead of the target.
    better = np.count_nonzero(scores > target_score)

    # Deterministic tie-break: lower item index ranks first.
    tied_before = np.count_nonzero(
        (scores == target_score)
        & (np.arange(n_items) < item_id)
    )

    return int(1 + better + tied_before)


# --------------------------------------------------
# 4. Evaluate every prepared profile
# --------------------------------------------------

evaluation_rows = []
tail_rank_rows = []

total_profiles = len(smoke_fold["support_samples"])

print("Prepared profiles:", f"{total_profiles:,}")
print("Inference methods:", 5)
print("Expected profile-method evaluations:", f"{total_profiles * 5:,}")
print("\nStarting evaluation...\n")

for profile_number, (key, profile_items) in enumerate(
    smoke_fold["support_samples"].items(),
    start=1,
):

    user_id, profile_size, sampling_mode, repeat_id = key

    user_id = int(user_id)
    profile_items = np.asarray(profile_items, dtype=np.int64)

    overall_targets = smoke_fold["overall_relevant"].get(
        user_id, set()
    )
    warm_targets = smoke_fold["warm_tail_relevant"].get(
        user_id, set()
    )
    cold_targets = smoke_fold["strict_cold_relevant"].get(
        user_id, set()
    )

    # The evaluation cohort consists of users with validation targets.
    if not overall_targets:
        continue

    # Leakage guard: selected movies cannot be validation targets.
    assert not (set(profile_items) & overall_targets)

    scores_by_method = score_all_methods(profile_items)

    for method_name, raw_scores in scores_by_method.items():

        scores = np.asarray(
            raw_scores,
            dtype=np.float64,
        ).copy()

        assert scores.shape == (n_items,)
        assert np.isfinite(scores).all()

        # Remove the visitor's selected movies from recommendations.
        scores[profile_items] = -np.inf

        top_items = stable_top_k(scores, EVAL_K)

        assert len(top_items) == EVAL_K
        assert not (set(top_items) & set(profile_items))

        overall_metrics = calculate_ranking_metrics(
            top_items,
            overall_targets,
        )

        warm_metrics = calculate_ranking_metrics(
            top_items,
            warm_targets,
        )

        cold_metrics = calculate_ranking_metrics(
            top_items,
            cold_targets,
        )

        evaluation_rows.append({
            "user_id": user_id,
            "profile_size": int(profile_size),
            "sampling_mode": sampling_mode,
            "repeat_id": int(repeat_id),
            "method": method_name,

            "n_overall_targets": len(overall_targets),
            "n_warm_tail_targets": len(warm_targets),
            "n_strict_cold_targets": len(cold_targets),

            "overall_ndcg": overall_metrics["ndcg"],
            "overall_recall": overall_metrics["recall"],
            "overall_hit": overall_metrics["hit"],

            "warm_tail_ndcg": warm_metrics["ndcg"],
            "warm_tail_recall": warm_metrics["recall"],
            "warm_tail_hit": warm_metrics["hit"],

            "strict_cold_ndcg": cold_metrics["ndcg"],
            "strict_cold_recall": cold_metrics["recall"],
            "strict_cold_hit": cold_metrics["hit"],
        })

        # Full-catalogue ranks for every warm-tail validation target.
        for item_id in warm_targets:

            item_id = int(item_id)
            rank = get_full_catalogue_rank(scores, item_id)

            tail_rank_rows.append({
                "user_id": user_id,
                "profile_size": int(profile_size),
                "sampling_mode": sampling_mode,
                "repeat_id": int(repeat_id),
                "method": method_name,
                "item_index": item_id,
                "rank": rank,
                "rank_percentile": 100.0 * (rank - 1) / (n_items - 1),
            })

    if profile_number % 250 == 0:
        elapsed = time.perf_counter() - eval_start
        print(
            f"Processed {profile_number:,}/{total_profiles:,} profiles | "
            f"{elapsed / 60:.1f} minutes"
        )


# --------------------------------------------------
# 5. Store detailed evaluation results
# --------------------------------------------------

evaluation_df = pd.DataFrame(evaluation_rows)
tail_ranks_df = pd.DataFrame(tail_rank_rows)

assert not evaluation_df.empty
assert not tail_ranks_df.empty

print("\nEvaluation completed.")
print("Profile-method rows:", f"{len(evaluation_df):,}")
print("Warm-tail rank observations:", f"{len(tail_ranks_df):,}")
print(
    "Total evaluation time:",
    f"{(time.perf_counter() - eval_start) / 60:.2f} minutes"
)


# --------------------------------------------------
# 6. Overall and conditional warm-tail metrics
# --------------------------------------------------

group_cols = [
    "profile_size",
    "sampling_mode",
    "method",
]

metric_summary = (
    evaluation_df
    .groupby(group_cols, as_index=False)
    .agg(
        profile_evaluations=("user_id", "size"),
        users_evaluated=("user_id", "nunique"),

        overall_NDCG_at_10=("overall_ndcg", "mean"),
        overall_Recall_at_10=("overall_recall", "mean"),
        overall_HitRate_at_10=("overall_hit", "mean"),

        warm_tail_NDCG_at_10=("warm_tail_ndcg", "mean"),
        warm_tail_Recall_at_10=("warm_tail_recall", "mean"),
        warm_tail_HitRate_at_10=("warm_tail_hit", "mean"),

        strict_cold_NDCG_at_10=("strict_cold_ndcg", "mean"),
        strict_cold_Recall_at_10=("strict_cold_recall", "mean"),
        strict_cold_HitRate_at_10=("strict_cold_hit", "mean"),
    )
)

# --------------------------------------------------
# 7. Count distinct warm-tail users reached
# --------------------------------------------------

per_user_hits = (
    evaluation_df
    .groupby(group_cols + ["user_id"], as_index=False)
    .agg(
        has_warm_tail_target=(
            "n_warm_tail_targets",
            "max",
        ),
        warm_tail_hit_any_repeat=(
            "warm_tail_hit",
            "max",
        ),
    )
)

per_user_hits["has_warm_tail_target"] = (
    per_user_hits["has_warm_tail_target"] > 0
)

per_user_hits["warm_tail_hit_any_repeat"] = (
    per_user_hits["warm_tail_hit_any_repeat"].fillna(0) > 0
)

tail_user_summary = (
    per_user_hits
    .groupby(group_cols, as_index=False)
    .apply(
        lambda x: pd.Series({
            "warm_tail_target_users": int(
                x["has_warm_tail_target"].sum()
            ),
            "warm_tail_users_hit_at_10_any_repeat": int(
                (
                    x["has_warm_tail_target"]
                    & x["warm_tail_hit_any_repeat"]
                ).sum()
            ),
            "warm_tail_user_discovery_rate_any_repeat": (
                (
                    x["has_warm_tail_target"]
                    & x["warm_tail_hit_any_repeat"]
                ).sum()
                / max(x["has_warm_tail_target"].sum(), 1)
            ),
        }),
        include_groups=False,
    )
    .reset_index(drop=True)
)

metric_summary = metric_summary.merge(
    tail_user_summary,
    on=group_cols,
    how="left",
)


# --------------------------------------------------
# 8. Full-catalogue warm-tail rank distributions
# --------------------------------------------------

tail_rank_summary = (
    tail_ranks_df
    .groupby(group_cols, as_index=False)
    .agg(
        warm_tail_rank_observations=("rank", "size"),
        mean_warm_tail_rank=("rank", "mean"),
        median_warm_tail_rank=("rank", "median"),
        p90_warm_tail_rank=("rank", lambda x: x.quantile(0.90)),

        pct_warm_tail_targets_top10=("rank", lambda x: (x <= 10).mean()),
        pct_warm_tail_targets_top100=("rank", lambda x: (x <= 100).mean()),
        pct_warm_tail_targets_top500=("rank", lambda x: (x <= 500).mean()),
        pct_warm_tail_targets_top1000=("rank", lambda x: (x <= 1000).mean()),
    )
)

# --------------------------------------------------
# 9. Evaluation cohort coverage
# --------------------------------------------------

coverage_rows = []

for profile_size in PROFILE_SIZES:
    for sampling_mode in SUPPORT_SAMPLING_MODES:

        subset = evaluation_df.loc[
            (evaluation_df["profile_size"] == profile_size)
            & (evaluation_df["sampling_mode"] == sampling_mode)
        ]

        evaluated_users = subset["user_id"].nunique()

        target_users = len(smoke_fold["overall_relevant"])

        coverage_rows.append({
            "profile_size": profile_size,
            "sampling_mode": sampling_mode,
            "heldout_users_with_targets": target_users,
            "users_evaluated": evaluated_users,
            "user_coverage_fraction": (
                evaluated_users / target_users
                if target_users else np.nan
            ),
            "warm_tail_target_users_evaluated": int(
                subset.loc[
                    subset["n_warm_tail_targets"] > 0,
                    "user_id",
                ].nunique()
            ),
        })

coverage_summary = pd.DataFrame(coverage_rows)


# --------------------------------------------------
# 10. Display results
# --------------------------------------------------

print("\n" + "=" * 80)
print("OVERALL AND WARM-TAIL METRICS")
print("=" * 80)

display(
    metric_summary.sort_values(
        ["profile_size", "sampling_mode", "overall_NDCG_at_10"],
        ascending=[True, True, False],
    ).round(5)
)

print("\n" + "=" * 80)
print("FULL-CATALOGUE WARM-TAIL TARGET RANKS")
print("=" * 80)

display(
    tail_rank_summary.sort_values(
        ["profile_size", "sampling_mode", "median_warm_tail_rank"]
    ).round(5)
)

print("\n" + "=" * 80)
print("EVALUATION COHORT COVERAGE")
print("=" * 80)

display(coverage_summary.round(4))

print("\nAll fold-0 evaluation checks completed.")


Prepared profiles: 13,804
Inference methods: 5
Expected profile-method evaluations: 69,020

Starting evaluation...

Processed 250/13,804 profiles | 0.0 minutes
Processed 500/13,804 profiles | 0.1 minutes
Processed 750/13,804 profiles | 0.1 minutes
Processed 1,000/13,804 profiles | 0.2 minutes
Processed 1,250/13,804 profiles | 0.2 minutes
Processed 1,500/13,804 profiles | 0.2 minutes
Processed 1,750/13,804 profiles | 0.3 minutes
Processed 2,000/13,804 profiles | 0.3 minutes
Processed 2,250/13,804 profiles | 0.4 minutes
Processed 2,500/13,804 profiles | 0.4 minutes
Processed 2,750/13,804 profiles | 0.4 minutes
Processed 3,000/13,804 profiles | 0.5 minutes
Processed 3,250/13,804 profiles | 0.5 minutes
Processed 3,500/13,804 profiles | 0.6 minutes
Processed 3,750/13,804 profiles | 0.6 minutes
Processed 4,000/13,804 profiles | 0.6 minutes
Processed 4,250/13,804 profiles | 0.7 minutes
Processed 4,500/13,804 profiles | 0.7 minutes
Processed 4,750/13,804 profiles | 0.8 minutes
Processed 5,000/

,profile_size,sampling_mode,method,profile_evaluations,users_evaluated,overall_NDCG_at_10,overall_Recall_at_10,overall_HitRate_at_10,warm_tail_NDCG_at_10,warm_tail_Recall_at_10,warm_tail_HitRate_at_10,strict_cold_NDCG_at_10,strict_cold_Recall_at_10,strict_cold_HitRate_at_10,warm_tail_target_users,warm_tail_users_hit_at_10_any_repeat,warm_tail_user_discovery_rate_any_repeat
1,3,popularity_biased,LightFM Content-v1 centroid,1812,906,0.02470,0.03783,0.10651,0.0,0.0,0.0,0.0,0.0,0.0,178.0,0.0,0.0
3,3,popularity_biased,LightFM Identity centroid,1812,906,0.02345,0.03770,0.10872,0.0,0.0,0.0,0.0,0.0,0.0,178.0,0.0,0.0
0,3,popularity_biased,ALS fold-in,1812,906,0.02259,0.03508,0.10044,0.0,0.0,0.0,0.0,0.0,0.0,178.0,0.0,0.0
2,3,popularity_biased,LightFM Content-v1 ridge,1812,906,0.02002,0.03206,0.09934,0.0,0.0,0.0,0.0,0.0,0.0,178.0,0.0,0.0
4,3,popularity_biased,LightFM Identity ridge,1812,906,0.01971,0.03142,0.09934,0.0,0.0,0.0,0.0,0.0,0.0,178.0,0.0,0.0
8,3,uniform,LightFM Identity centroid,1812,906,0.02713,0.04163,0.11865,0.0,0.0,0.0,0.0,0.0,0.0,178.0,0.0,0.0
6,3,uniform,LightFM Content-v1 centroid,1812,906,0.02640,0.04059,0.11313,0.0,0.0,0.0,0.0,0.0,0.0,178.0,0.0,0.0
5,3,uniform,ALS fold-in,1812,906,0.02636,0.04186,0.11479,0.0,0.0,0.0,0.0,0.0,0.0,178.0,0.0,0.0
7,3,uniform,LightFM Content-v1 ridge,1812,906,0.02066,0.03349,0.09934,0.0,0.0,0.0,0.0,0.0,0.0,178.0,0.0,0.0
9,3,uniform,LightFM Identity ridge,1812,906,0.01979,0.03157,0.09879,0.0,0.0,0.0,0.0,0.0,0.0,178.0,0.0,0.0



FULL-CATALOGUE WARM-TAIL TARGET RANKS


,profile_size,sampling_mode,method,warm_tail_rank_observations,mean_warm_tail_rank,median_warm_tail_rank,p90_warm_tail_rank,pct_warm_tail_targets_top10,pct_warm_tail_targets_top100,pct_warm_tail_targets_top500,pct_warm_tail_targets_top1000
3,3,popularity_biased,LightFM Identity centroid,562,6312.74377,5338.5,12308.5,0.0,0.00000,0.00712,0.02847
1,3,popularity_biased,LightFM Content-v1 centroid,562,6754.25623,5546.0,14191.0,0.0,0.00000,0.01423,0.06406
0,3,popularity_biased,ALS fold-in,562,6898.60854,5590.0,14168.9,0.0,0.00000,0.00178,0.04270
4,3,popularity_biased,LightFM Identity ridge,562,6957.90925,6303.0,12427.7,0.0,0.00000,0.00000,0.00356
2,3,popularity_biased,LightFM Content-v1 ridge,562,9091.63701,8517.5,15660.3,0.0,0.00000,0.00178,0.00178
6,3,uniform,LightFM Content-v1 centroid,562,5067.39324,3909.5,10879.1,0.0,0.00534,0.04448,0.11566
8,3,uniform,LightFM Identity centroid,562,5051.80961,4237.5,9758.2,0.0,0.00000,0.01423,0.05694
5,3,uniform,ALS fold-in,562,6189.79715,4845.5,13790.3,0.0,0.00000,0.00356,0.06228
9,3,uniform,LightFM Identity ridge,562,5869.43060,5285.0,10213.2,0.0,0.00000,0.00000,0.00890
7,3,uniform,LightFM Content-v1 ridge,562,7963.46441,7392.5,14354.9,0.0,0.00000,0.00356,0.00534



EVALUATION COHORT COVERAGE


,profile_size,sampling_mode,heldout_users_with_targets,users_evaluated,user_coverage_fraction,warm_tail_target_users_evaluated
0,3,uniform,906,906,1.0000,178
1,3,popularity_biased,906,906,1.0000,178
2,5,uniform,906,906,1.0000,178
3,5,popularity_biased,906,906,1.0000,178
4,10,uniform,906,893,0.9857,178
5,10,popularity_biased,906,893,0.9857,178
6,20,uniform,906,746,0.8234,164
7,20,popularity_biased,906,746,0.8234,164



All fold-0 evaluation checks completed.


In [12]:

# ============================================================
# SHARED FIVE-FOLD EVALUATION HELPERS
# ============================================================

required_objects = [
    "development_fold_users",
    "_build_fold_context",
    "fit_fold_implicit_als",
    "fold_in_als_profile",
    "lightfm_centroid_fold_in",
    "lightfm_ridge_fold_in",
    "smoke_fold",
    "Q_als_smoke",
    "lightfm_smoke_representations",
    "LIGHTFM_RIDGE_LAMBDA_SMOKE",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Missing notebook variables/functions. Run the earlier setup, "
        "fold-preparation, training and inference cells first: "
        + ", ".join(missing_objects)
    )

assert set(development_fold_users) == set(range(N_FOLDS))
assert len(smoke_fold["support_samples"]) > 0
assert Q_als_smoke.shape == (n_items, 64)
assert LIGHTFM_RIDGE_LAMBDA_SMOKE == 1.0

CV_TOP_K = 10
CV_SUPPORT_REPEATS = SMOKE_SUPPORT_REPEATS
CV_METHOD_NAMES = [
    "ALS fold-in",
    "LightFM Identity centroid",
    "LightFM Identity ridge",
    "LightFM Content-v1 centroid",
    "LightFM Content-v1 ridge",
]


def _score_cv_methods(profile_items, Q_als, lightfm_reps):
    """Return full-catalogue scores from all five frozen methods."""

    profile_items = np.asarray(profile_items, dtype=np.int64)

    # ALS anonymous-user fold-in
    als_user = fold_in_als_profile(profile_items, Q_als)
    als_scores = Q_als @ als_user

    results = {
        "ALS fold-in": np.asarray(als_scores, dtype=np.float64)
    }

    # LightFM Identity and Content-v1
    for feature_name in ["Identity", "Content-v1"]:

        reps = lightfm_reps[feature_name]
        item_vectors = reps["item_vectors"]
        item_bias = reps["item_bias"]

        # Centroid inference
        centroid_user = lightfm_centroid_fold_in(
            profile_items,
            item_vectors,
        )

        centroid_scores = (
            item_vectors @ centroid_user + item_bias
        )

        results[
            f"LightFM {feature_name} centroid"
        ] = np.asarray(centroid_scores, dtype=np.float64)

        # Deterministic ridge-style inference
        ridge_user = lightfm_ridge_fold_in(
            profile_items,
            item_vectors,
            ridge=LIGHTFM_RIDGE_LAMBDA_SMOKE,
        )

        ridge_scores = (
            item_vectors @ ridge_user + item_bias
        )

        results[
            f"LightFM {feature_name} ridge"
        ] = np.asarray(ridge_scores, dtype=np.float64)

    assert set(results) == set(CV_METHOD_NAMES)

    for method_name, scores in results.items():
        assert scores.shape == (n_items,), method_name
        assert np.isfinite(scores).all(), method_name

    return results


def _cv_ranking_metrics(top_items, relevant_items, k=CV_TOP_K):

    relevant_items = set(map(int, relevant_items))

    if not relevant_items:
        return {
            "ndcg": np.nan,
            "recall": np.nan,
            "hit": np.nan,
        }

    hit_ranks = [
        rank
        for rank, item in enumerate(top_items, start=1)
        if int(item) in relevant_items
    ]

    dcg = sum(
        1.0 / np.log2(rank + 1)
        for rank in hit_ranks
    )

    ideal_count = min(len(relevant_items), k)

    idcg = sum(
        1.0 / np.log2(rank + 1)
        for rank in range(1, ideal_count + 1)
    )

    return {
        "ndcg": dcg / idcg if idcg else 0.0,
        "recall": len(hit_ranks) / len(relevant_items),
        "hit": float(len(hit_ranks) > 0),
    }


def _evaluate_cv_fold(fold_id, fold_ctx, Q_als, lightfm_reps):
    """
    Evaluate one fold across all sampled profiles and five methods.

    One full-catalogue sort per method/profile gives both top-10
    recommendations and exact deterministic ranks for warm-tail targets.
    """

    evaluation_rows = []
    tail_rank_rows = []

    item_ids = np.arange(n_items, dtype=np.int64)

    support_samples = fold_ctx["support_samples"]

    print(
        f"Evaluating fold {fold_id}: "
        f"{len(support_samples):,} sampled profiles"
    )

    for profile_number, (key, profile_items) in enumerate(
        support_samples.items(),
        start=1,
    ):

        user_id, profile_size, sampling_mode, repeat_id = key

        user_id = int(user_id)
        profile_items = np.asarray(profile_items, dtype=np.int64)

        overall_targets = fold_ctx["overall_relevant"].get(
            user_id, set()
        )
        warm_targets = fold_ctx["warm_tail_relevant"].get(
            user_id, set()
        )
        cold_targets = fold_ctx["strict_cold_relevant"].get(
            user_id, set()
        )

        assert overall_targets, f"Missing validation targets for user {user_id}"
        assert not (set(profile_items) & overall_targets), (
            f"Support/target leakage for user {user_id}"
        )

        scores_by_method = _score_cv_methods(
            profile_items,
            Q_als,
            lightfm_reps,
        )

        for method_name, raw_scores in scores_by_method.items():

            # Exclude only the selected profile movies.
            scores = raw_scores.copy()
            scores[profile_items] = -np.inf

            # Deterministic ranking: score descending, item index ascending.
            rank_order = np.lexsort((item_ids, -scores))

            top_items = rank_order[:CV_TOP_K]

            assert len(top_items) == CV_TOP_K
            assert not (set(top_items) & set(profile_items))

            # Construct a full-catalogue rank lookup.
            rank_by_item = np.empty(n_items, dtype=np.int32)
            rank_by_item[rank_order] = np.arange(
                1, n_items + 1, dtype=np.int32
            )

            overall_metrics = _cv_ranking_metrics(
                top_items, overall_targets
            )
            warm_metrics = _cv_ranking_metrics(
                top_items, warm_targets
            )
            cold_metrics = _cv_ranking_metrics(
                top_items, cold_targets
            )

            evaluation_rows.append({
                "fold_id": int(fold_id),
                "user_id": user_id,
                "profile_size": int(profile_size),
                "sampling_mode": sampling_mode,
                "repeat_id": int(repeat_id),
                "method": method_name,

                "n_overall_targets": len(overall_targets),
                "n_warm_tail_targets": len(warm_targets),
                "n_strict_cold_targets": len(cold_targets),

                "overall_ndcg": overall_metrics["ndcg"],
                "overall_recall": overall_metrics["recall"],
                "overall_hit": overall_metrics["hit"],

                "warm_tail_ndcg": warm_metrics["ndcg"],
                "warm_tail_recall": warm_metrics["recall"],
                "warm_tail_hit": warm_metrics["hit"],

                "strict_cold_ndcg": cold_metrics["ndcg"],
                "strict_cold_recall": cold_metrics["recall"],
                "strict_cold_hit": cold_metrics["hit"],
            })

            # Record the exact full-catalogue rank of every warm-tail target.
            for item_id in warm_targets:

                item_id = int(item_id)
                rank = int(rank_by_item[item_id])

                tail_rank_rows.append({
                    "fold_id": int(fold_id),
                    "user_id": user_id,
                    "profile_size": int(profile_size),
                    "sampling_mode": sampling_mode,
                    "repeat_id": int(repeat_id),
                    "method": method_name,
                    "item_index": item_id,
                    "rank": rank,
                    "rank_percentile": (
                        100.0 * (rank - 1) / (n_items - 1)
                    ),
                })

        if profile_number % 1000 == 0:
            print(
                f"  Fold {fold_id}: "
                f"{profile_number:,}/{len(support_samples):,} profiles"
            )

    evaluation_df_fold = pd.DataFrame(evaluation_rows)
    tail_ranks_df_fold = pd.DataFrame(tail_rank_rows)

    assert not evaluation_df_fold.empty
    assert set(evaluation_df_fold["method"].unique()) == set(CV_METHOD_NAMES)
    assert evaluation_df_fold["fold_id"].nunique() == 1
    assert evaluation_df_fold["fold_id"].iloc[0] == fold_id

    print(
        f"Fold {fold_id} evaluation complete | "
        f"profile-method rows: {len(evaluation_df_fold):,} | "
        f"warm-tail rank rows: {len(tail_ranks_df_fold):,}"
    )

    return evaluation_df_fold, tail_ranks_df_fold


In [13]:

# ============================================================
# RUN THE COMPLETE FIVE-FOLD DEVELOPMENT EXPERIMENT
# Fold 0 is reused; folds 1-4 are trained from scratch.
# ============================================================

import gc
import time
from lightfm import LightFM

cv_run_start = time.perf_counter()

evaluation_parts = []
tail_rank_parts = []
fit_audit_rows = []


# ------------------------------------------------------------
# FOLD 0: reuse the models already trained in the smoke test
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FOLD 0 — REUSING EXISTING TRAINED MODELS")
print("=" * 75)

fold0_eval, fold0_tail = _evaluate_cv_fold(
    fold_id=0,
    fold_ctx=smoke_fold,
    Q_als=Q_als_smoke,
    lightfm_reps=lightfm_smoke_representations,
)

evaluation_parts.append(fold0_eval)
tail_rank_parts.append(fold0_tail)

fit_audit_rows.append({
    "fold_id": 0,
    "status": "reused_smoke_models",
    "als_training_seconds": als_smoke["training_time_seconds"],
    "identity_training_seconds": np.nan,
    "content_v1_training_seconds": np.nan,
    "positive_interactions": als_smoke["positive_interactions"],
    "training_rating_rows": len(smoke_fold["fold_train"]),
    "training_users": smoke_fold["fold_train"]["u"].nunique(),
    "warm_tail_catalogue_items": int(
        smoke_fold["warm_tail_mask"].sum()
    ),
    "warm_tail_validation_pairs": smoke_fold["summary"][
        "warm_tail_validation_pairs"
    ],
    "sampled_profiles": len(smoke_fold["support_samples"]),
})


# ------------------------------------------------------------
# LIGHTFM FOLD TRAINING HELPER
# ------------------------------------------------------------

def _fit_lightfm_fold_representation(
    fold_id,
    fold_ctx,
    feature_name,
    item_features,
):
    """Fit one frozen LightFM configuration and extract item vectors."""

    model = LightFM(
        no_components=LIGHTFM_COMPONENTS,
        loss="warp",
        learning_rate=LIGHTFM_LEARNING_RATE,
        item_alpha=LIGHTFM_ITEM_ALPHA,
        user_alpha=LIGHTFM_USER_ALPHA,
        max_sampled=LIGHTFM_MAX_SAMPLED,
        random_state=SEED,
    )

    start = time.perf_counter()

    model.fit(
        interactions=fold_ctx["interactions"],
        item_features=item_features,
        epochs=LIGHTFM_EPOCHS,
        num_threads=NUM_THREADS,
        verbose=True,
    )

    elapsed = time.perf_counter() - start

    item_bias, item_vectors = model.get_item_representations(
        item_features
    )

    user_bias, user_vectors = model.get_user_representations()

    assert item_vectors.shape == (
        n_items,
        LIGHTFM_COMPONENTS,
    )
    assert item_bias.shape == (n_items,)

    assert np.isfinite(item_vectors).all()
    assert np.isfinite(item_bias).all()

    # Check that the extracted scoring representation still matches
    # the official LightFM scorer for a real model-training user.
    check_user = int(
        fold_ctx["fold_train"]["u"].iloc[0]
    )

    check_items = np.unique(
        np.array([0, 1, 17, n_items - 1], dtype=np.int32)
    )

    direct_scores = (
        user_vectors[check_user] @ item_vectors[check_items].T
        + user_bias[check_user]
        + item_bias[check_items]
    )

    official_scores = model.predict(
        user_ids=np.full(
            len(check_items),
            check_user,
            dtype=np.int32,
        ),
        item_ids=check_items,
        item_features=item_features,
        num_threads=NUM_THREADS,
    )

    assert np.allclose(
        direct_scores,
        official_scores,
        rtol=1e-4,
        atol=1e-5,
    ), (
        f"Fold {fold_id} {feature_name}: "
        "direct scoring does not match LightFM predict()"
    )

    result = {
        "item_vectors": item_vectors.astype(np.float32, copy=True),
        "item_bias": item_bias.astype(np.float32, copy=True),
        "training_time_seconds": elapsed,
    }

    del model, user_bias, user_vectors
    gc.collect()

    return result


# ------------------------------------------------------------
# FOLDS 1-4: build context, train three models, evaluate
# ------------------------------------------------------------

for fold_id in range(1, N_FOLDS):

    print("\n" + "=" * 75)
    print(f"FOLD {fold_id} — BUILDING LEAKAGE-SAFE CONTEXT")
    print("=" * 75)

    fold_start = time.perf_counter()

    fold_ctx = _build_fold_context(
        fold_id,
        support_repeats=CV_SUPPORT_REPEATS,
    )

    # ----------------------------
    # Leakage and cohort assertions
    # ----------------------------

    heldout_set = set(map(int, fold_ctx["heldout_users"]))
    pilot_set = set(map(int, pilot_confirmation_users))
    excluded_set = set(map(int, fold_ctx["excluded_users"]))

    assert heldout_set.isdisjoint(pilot_set)
    assert excluded_set == heldout_set | pilot_set

    assert not fold_ctx["fold_train"]["u"].isin(
        excluded_set
    ).any()

    assert fold_ctx["interactions"].tocsr()[
        fold_ctx["excluded_users"]
    ].nnz == 0

    assert (
        len(fold_ctx["fold_positive"])
        == fold_ctx["interactions"].nnz
    )

    print("Fold context:")
    display(
        pd.Series(
            fold_ctx["summary"],
            name="value",
        ).to_frame()
    )

    # ----------------------------
    # 1. ALS
    # ----------------------------

    print(f"\nFitting ALS for fold {fold_id}...")

    als_fit = fit_fold_implicit_als(
        fold_ctx["fold_train"],
        k=ALS_FACTORS,
        alpha=ALS_ALPHA,
        lam=ALS_LAMBDA,
        n_iter=ALS_ITERATIONS,
        seed=SEED,
    )

    Q_als = als_fit["Q"]

    assert Q_als.shape == (n_items, ALS_FACTORS)
    assert np.isfinite(Q_als).all()

    # Strict-cold items must have no ALS item factors.
    item_has_als_factors = np.abs(Q_als).sum(axis=1) > 0

    assert np.array_equal(
        ~item_has_als_factors,
        fold_ctx["strict_cold_mask"],
    )

    print(
        f"ALS complete: {als_fit['training_time_seconds']:.1f}s | "
        f"Q shape: {Q_als.shape}"
    )

    # ----------------------------
    # 2. LightFM Identity
    # ----------------------------

    print(f"\nFitting LightFM Identity for fold {fold_id}...")

    identity_fit = _fit_lightfm_fold_representation(
        fold_id,
        fold_ctx,
        "Identity",
        identity_item_features,
    )

    print(
        "Identity complete:",
        f"{identity_fit['training_time_seconds']:.1f}s"
    )

    # ----------------------------
    # 3. LightFM Content-v1
    # ----------------------------

    print(f"\nFitting LightFM Content-v1 for fold {fold_id}...")

    content_fit = _fit_lightfm_fold_representation(
        fold_id,
        fold_ctx,
        "Content-v1",
        content_v1_item_features,
    )

    print(
        "Content-v1 complete:",
        f"{content_fit['training_time_seconds']:.1f}s"
    )

    fold_lightfm_reps = {
        "Identity": {
            "item_vectors": identity_fit["item_vectors"],
            "item_bias": identity_fit["item_bias"],
        },
        "Content-v1": {
            "item_vectors": content_fit["item_vectors"],
            "item_bias": content_fit["item_bias"],
        },
    }

    # ----------------------------
    # Evaluate this fold
    # ----------------------------

    print(f"\nEvaluating fold {fold_id}...")

    fold_eval, fold_tail = _evaluate_cv_fold(
        fold_id=fold_id,
        fold_ctx=fold_ctx,
        Q_als=Q_als,
        lightfm_reps=fold_lightfm_reps,
    )

    evaluation_parts.append(fold_eval)
    tail_rank_parts.append(fold_tail)

    # ----------------------------
    # Save compact training audit
    # ----------------------------

    fit_audit_rows.append({
        "fold_id": fold_id,
        "status": "trained",
        "als_training_seconds": als_fit["training_time_seconds"],
        "identity_training_seconds": identity_fit["training_time_seconds"],
        "content_v1_training_seconds": content_fit["training_time_seconds"],
        "positive_interactions": als_fit["positive_interactions"],
        "training_rating_rows": len(fold_ctx["fold_train"]),
        "training_users": fold_ctx["fold_train"]["u"].nunique(),
        "warm_tail_catalogue_items": int(
            fold_ctx["warm_tail_mask"].sum()
        ),
        "warm_tail_validation_pairs": fold_ctx["summary"][
            "warm_tail_validation_pairs"
        ],
        "sampled_profiles": len(fold_ctx["support_samples"]),
        "total_fold_seconds": time.perf_counter() - fold_start,
    })

    print(
        f"\nFold {fold_id} completed in "
        f"{(time.perf_counter() - fold_start) / 60:.2f} minutes."
    )

    # Free fold-specific objects before starting the next fold.
    del (
        fold_ctx,
        als_fit,
        Q_als,
        item_has_als_factors,
        identity_fit,
        content_fit,
        fold_lightfm_reps,
        fold_eval,
        fold_tail,
    )

    gc.collect()


# ------------------------------------------------------------
# COMBINE FIVE-FOLD OUTPUTS
# ------------------------------------------------------------

evaluation_all_folds = pd.concat(
    evaluation_parts,
    ignore_index=True,
)

tail_ranks_all_folds = pd.concat(
    tail_rank_parts,
    ignore_index=True,
)

fold_fit_summary = pd.DataFrame(fit_audit_rows)

assert set(evaluation_all_folds["fold_id"].unique()) == set(range(N_FOLDS))
assert set(tail_ranks_all_folds["fold_id"].unique()) == set(range(N_FOLDS))

assert set(evaluation_all_folds["method"].unique()) == set(
    CV_METHOD_NAMES
)

# Every held-out user must belong to exactly one development fold.
assert all(
    development_fold_id_by_user[int(user_id)] == int(fold_id)
    for fold_id, user_id in (
        evaluation_all_folds[["fold_id", "user_id"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )
)

print("\n" + "=" * 75)
print("FIVE-FOLD DEVELOPMENT EXPERIMENT COMPLETE")
print("=" * 75)

print("Evaluation rows:", f"{len(evaluation_all_folds):,}")
print("Warm-tail rank observations:", f"{len(tail_ranks_all_folds):,}")
print("Users evaluated:", evaluation_all_folds["user_id"].nunique())
print(
    "Elapsed time:",
    f"{(time.perf_counter() - cv_run_start) / 60:.2f} minutes"
)

display(fold_fit_summary.round(2))



FOLD 0 — REUSING EXISTING TRAINED MODELS
Evaluating fold 0: 13,804 sampled profiles
  Fold 0: 1,000/13,804 profiles
  Fold 0: 2,000/13,804 profiles
  Fold 0: 3,000/13,804 profiles
  Fold 0: 4,000/13,804 profiles
  Fold 0: 5,000/13,804 profiles
  Fold 0: 6,000/13,804 profiles
  Fold 0: 7,000/13,804 profiles
  Fold 0: 8,000/13,804 profiles
  Fold 0: 9,000/13,804 profiles
  Fold 0: 10,000/13,804 profiles
  Fold 0: 11,000/13,804 profiles
  Fold 0: 12,000/13,804 profiles
  Fold 0: 13,000/13,804 profiles
Fold 0 evaluation complete | profile-method rows: 69,020 | warm-tail rank rows: 22,100

FOLD 1 — BUILDING LEAKAGE-SAFE CONTEXT
Fold context:


,value
fold_id,1
heldout_development_users,906
pilot_confirmation_users_excluded,1132
model_training_users_with_any_rows,18055
model_training_rating_rows,2530213
model_training_positive_pairs,1244919
catalogue_items_with_positive_history,15593
warm_tail_catalogue_items,10461
strict_cold_catalogue_items,1648
head_catalogue_items,5132



Fitting ALS for fold 1...
ALS iteration 01/10 | 4.0s
ALS iteration 02/10 | 8.1s
ALS iteration 04/10 | 16.5s
ALS iteration 06/10 | 24.6s
ALS iteration 08/10 | 32.8s
ALS iteration 10/10 | 41.6s
ALS complete: 41.6s | Q shape: (17241, 64)

Fitting LightFM Identity for fold 1...
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Identity complete: 21.2s

Fitting LightFM Content-v1 for fold 1...
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Content-v1 complete: 31.9s

Evaluating fold 1...
Evaluating fold 1: 13,812 sampled profiles
  Fold 1: 1,000/13,812 profiles
  Fold 1: 2,000/13,812 profiles
  Fold 1: 3,000/13,812 profiles
  Fold 1: 4,000/13,812 profiles
  Fold 1: 5,000/13,812 profiles
  Fold 1: 6,000/13,812 profiles
  Fold 1: 7,000/13,812 profiles
  Fold 1: 8,000/13,812 profiles
  Fold 1: 9,000/13,812 profiles
  Fold 1: 10,000/13,812 pro

,value
fold_id,2
heldout_development_users,905
pilot_confirmation_users_excluded,1132
model_training_users_with_any_rows,18056
model_training_rating_rows,2514835
model_training_positive_pairs,1238195
catalogue_items_with_positive_history,15525
warm_tail_catalogue_items,10430
strict_cold_catalogue_items,1716
head_catalogue_items,5095



Fitting ALS for fold 2...
ALS iteration 01/10 | 4.1s
ALS iteration 02/10 | 8.8s
ALS iteration 04/10 | 16.3s
ALS iteration 06/10 | 24.7s
ALS iteration 08/10 | 32.3s
ALS iteration 10/10 | 39.9s
ALS complete: 39.9s | Q shape: (17241, 64)

Fitting LightFM Identity for fold 2...
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Identity complete: 19.9s

Fitting LightFM Content-v1 for fold 2...
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Content-v1 complete: 34.8s

Evaluating fold 2...
Evaluating fold 2: 13,876 sampled profiles
  Fold 2: 1,000/13,876 profiles
  Fold 2: 2,000/13,876 profiles
  Fold 2: 3,000/13,876 profiles
  Fold 2: 4,000/13,876 profiles
  Fold 2: 5,000/13,876 profiles
  Fold 2: 6,000/13,876 profiles
  Fold 2: 7,000/13,876 profiles
  Fold 2: 8,000/13,876 profiles
  Fold 2: 9,000/13,876 profiles
  Fold 2: 10,000/13,876 pro

,value
fold_id,3
heldout_development_users,905
pilot_confirmation_users_excluded,1132
model_training_users_with_any_rows,18056
model_training_rating_rows,2528523
model_training_positive_pairs,1245566
catalogue_items_with_positive_history,15635
warm_tail_catalogue_items,10491
strict_cold_catalogue_items,1606
head_catalogue_items,5144



Fitting ALS for fold 3...
ALS iteration 01/10 | 3.7s
ALS iteration 02/10 | 7.4s
ALS iteration 04/10 | 15.3s
ALS iteration 06/10 | 24.0s
ALS iteration 08/10 | 31.7s
ALS iteration 10/10 | 39.3s
ALS complete: 39.3s | Q shape: (17241, 64)

Fitting LightFM Identity for fold 3...
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Identity complete: 22.5s

Fitting LightFM Content-v1 for fold 3...
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Content-v1 complete: 36.8s

Evaluating fold 3...
Evaluating fold 3: 13,808 sampled profiles
  Fold 3: 1,000/13,808 profiles
  Fold 3: 2,000/13,808 profiles
  Fold 3: 3,000/13,808 profiles
  Fold 3: 4,000/13,808 profiles
  Fold 3: 5,000/13,808 profiles
  Fold 3: 6,000/13,808 profiles
  Fold 3: 7,000/13,808 profiles
  Fold 3: 8,000/13,808 profiles
  Fold 3: 9,000/13,808 profiles
  Fold 3: 10,000/13,808 pro

,value
fold_id,4
heldout_development_users,905
pilot_confirmation_users_excluded,1132
model_training_users_with_any_rows,18056
model_training_rating_rows,2516448
model_training_positive_pairs,1244006
catalogue_items_with_positive_history,15519
warm_tail_catalogue_items,10384
strict_cold_catalogue_items,1722
head_catalogue_items,5135



Fitting ALS for fold 4...
ALS iteration 01/10 | 3.9s
ALS iteration 02/10 | 7.9s
ALS iteration 04/10 | 15.9s
ALS iteration 06/10 | 23.7s
ALS iteration 08/10 | 31.9s
ALS iteration 10/10 | 39.4s
ALS complete: 39.4s | Q shape: (17241, 64)

Fitting LightFM Identity for fold 4...
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Identity complete: 22.4s

Fitting LightFM Content-v1 for fold 4...
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Content-v1 complete: 33.9s

Evaluating fold 4...
Evaluating fold 4: 13,872 sampled profiles
  Fold 4: 1,000/13,872 profiles
  Fold 4: 2,000/13,872 profiles
  Fold 4: 3,000/13,872 profiles
  Fold 4: 4,000/13,872 profiles
  Fold 4: 5,000/13,872 profiles
  Fold 4: 6,000/13,872 profiles
  Fold 4: 7,000/13,872 profiles
  Fold 4: 8,000/13,872 profiles
  Fold 4: 9,000/13,872 profiles
  Fold 4: 10,000/13,872 pro

,fold_id,status,als_training_seconds,identity_training_seconds,content_v1_training_seconds,positive_interactions,training_rating_rows,training_users,warm_tail_catalogue_items,warm_tail_validation_pairs,sampled_profiles,total_fold_seconds
0,0,reused_smoke_models,56.42,NaN,NaN,1248910,2533620,18055,10469,281,13804,NaN
1,1,trained,41.60,21.18,31.86,1244919,2530213,18055,10461,316,13812,344.38
2,2,trained,39.94,19.95,34.78,1238195,2514835,18056,10430,363,13876,340.11
3,3,trained,39.32,22.49,36.78,1245566,2528523,18056,10491,311,13808,366.47
4,4,trained,39.41,22.43,33.88,1244006,2516448,18056,10384,308,13872,334.17


In [14]:

# ============================================================
# FINAL FIVE-FOLD DEVELOPMENT RESULTS
# ============================================================

# 1. Sanity checks
assert set(evaluation_all_folds["fold_id"].unique()) == set(range(5))
assert evaluation_all_folds["user_id"].nunique() == len(development_users)
assert len(evaluation_all_folds) == 345860
assert len(tail_ranks_all_folds) == 124720

keys = [
    "fold_id", "user_id", "profile_size",
    "sampling_mode", "repeat_id", "method"
]
assert not evaluation_all_folds.duplicated(keys).any()

repeat_counts = (
    evaluation_all_folds
    .groupby([
        "fold_id", "user_id", "profile_size",
        "sampling_mode", "method"
    ])["repeat_id"]
    .nunique()
)
assert (repeat_counts == 2).all()


# 2. Average repeats within each user first
user_cols = [
    "fold_id", "user_id", "profile_size",
    "sampling_mode", "method"
]

evaluation_user_level = (
    evaluation_all_folds
    .groupby(user_cols, as_index=False)
    .agg(
        n_overall_targets=("n_overall_targets", "max"),
        n_warm_tail_targets=("n_warm_tail_targets", "max"),
        n_strict_cold_targets=("n_strict_cold_targets", "max"),

        overall_ndcg=("overall_ndcg", "mean"),
        overall_recall=("overall_recall", "mean"),
        overall_hit_rate=("overall_hit", "mean"),

        warm_tail_ndcg=("warm_tail_ndcg", "mean"),
        warm_tail_recall=("warm_tail_recall", "mean"),
        warm_tail_hit_per_profile=("warm_tail_hit", "mean"),
        warm_tail_hit_any_repeat=("warm_tail_hit", "max"),

        strict_cold_ndcg=("strict_cold_ndcg", "mean"),
        strict_cold_recall=("strict_cold_recall", "mean"),
        strict_cold_hit_rate=("strict_cold_hit", "mean"),
    )
)

evaluation_user_level["has_warm_tail_target"] = (
    evaluation_user_level["n_warm_tail_targets"] > 0
)

evaluation_user_level["warm_tail_discovery"] = np.where(
    evaluation_user_level["has_warm_tail_target"],
    evaluation_user_level["warm_tail_hit_any_repeat"],
    np.nan,
)

evaluation_user_level["has_strict_cold_target"] = (
    evaluation_user_level["n_strict_cold_targets"] > 0
)


# 3. User-level metric aggregation
def summarize_users(data, group_cols):
    return (
        data.groupby(group_cols, as_index=False)
        .agg(
            users_evaluated=("user_id", "nunique"),

            overall_NDCG_at_10=("overall_ndcg", "mean"),
            overall_Recall_at_10=("overall_recall", "mean"),
            overall_HitRate_at_10=("overall_hit_rate", "mean"),

            warm_tail_target_users=("has_warm_tail_target", "sum"),
            warm_tail_NDCG_at_10=("warm_tail_ndcg", "mean"),
            warm_tail_Recall_at_10=("warm_tail_recall", "mean"),
            warm_tail_HitRate_per_profile=(
                "warm_tail_hit_per_profile", "mean"
            ),
            warm_tail_users_discovered=(
                "warm_tail_discovery", "sum"
            ),
            warm_tail_discovery_rate_any_repeat=(
                "warm_tail_discovery", "mean"
            ),

            strict_cold_target_users=(
                "has_strict_cold_target", "sum"
            ),
            strict_cold_Recall_at_10=("strict_cold_recall", "mean"),
        )
    )


pooled_metrics = summarize_users(
    evaluation_user_level,
    ["profile_size", "sampling_mode", "method"],
)

fold_metrics = summarize_users(
    evaluation_user_level,
    ["fold_id", "profile_size", "sampling_mode", "method"],
)


# 4. Warm-tail full-catalogue rank summaries
def summarize_ranks(data, group_cols):
    return (
        data.groupby(group_cols, as_index=False)
        .agg(
            rank_observations=("rank", "size"),
            median_rank=("rank", "median"),
            mean_rank=("rank", "mean"),
            p90_rank=("rank", lambda x: x.quantile(0.90)),
            targets_top10=("rank", lambda x: (x <= 10).mean()),
            targets_top100=("rank", lambda x: (x <= 100).mean()),
            targets_top500=("rank", lambda x: (x <= 500).mean()),
            targets_top1000=("rank", lambda x: (x <= 1000).mean()),
        )
    )


pooled_ranks = summarize_ranks(
    tail_ranks_all_folds,
    ["profile_size", "sampling_mode", "method"],
)

fold_ranks = summarize_ranks(
    tail_ranks_all_folds,
    ["fold_id", "profile_size", "sampling_mode", "method"],
)


# 5. Compare Content-v1 centroid directly against ALS
comparison = fold_metrics.loc[
    fold_metrics["method"].isin([
        "ALS fold-in",
        "LightFM Content-v1 centroid",
    ])
].copy()

delta_metrics = [
    "overall_NDCG_at_10",
    "overall_Recall_at_10",
    "warm_tail_Recall_at_10",
    "warm_tail_HitRate_per_profile",
    "warm_tail_discovery_rate_any_repeat",
]

comparison_delta = comparison.pivot(
    index=["fold_id", "profile_size", "sampling_mode"],
    columns="method",
    values=delta_metrics,
)

delta_rows = []
for metric in delta_metrics:
    temp = comparison_delta[metric].reset_index()
    temp["metric"] = metric
    temp["content_minus_als"] = (
        temp["LightFM Content-v1 centroid"]
        - temp["ALS fold-in"]
    )
    delta_rows.append(
        temp[
            [
                "fold_id", "profile_size",
                "sampling_mode", "metric",
                "content_minus_als"
            ]
        ]
    )

fold_deltas = pd.concat(delta_rows, ignore_index=True)


# 6. Display
print("\nA. POOLED FIVE-FOLD METRICS — ALL METHODS")
display(
    pooled_metrics.sort_values(
        ["profile_size", "sampling_mode", "overall_NDCG_at_10"],
        ascending=[True, True, False],
    ).round(5)
)

print("\nB. FOLD-BY-FOLD METRICS — ALS VS CONTENT CENTROID")
display(
    comparison.sort_values(
        ["profile_size", "sampling_mode", "method", "fold_id"]
    ).round(5)
)

print("\nC. POOLED WARM-TAIL TARGET RANKS")
display(
    pooled_ranks.sort_values(
        ["profile_size", "sampling_mode", "median_rank"]
    ).round(5)
)

print("\nD. FOLD-BY-FOLD WARM-TAIL RANKS — ALS VS CONTENT CENTROID")
display(
    fold_ranks.loc[
        fold_ranks["method"].isin([
            "ALS fold-in",
            "LightFM Content-v1 centroid",
        ])
    ].sort_values(
        ["profile_size", "sampling_mode", "method", "fold_id"]
    ).round(5)
)

print("\nE. CONTENT CENTROID MINUS ALS — FOLD-LEVEL DIFFERENCES")
display(
    fold_deltas.sort_values(
        ["profile_size", "sampling_mode", "metric", "fold_id"]
    ).round(6)
)

print("\nF. TRAINING AUDIT")
display(fold_fit_summary.round(2))



A. POOLED FIVE-FOLD METRICS — ALL METHODS


,profile_size,sampling_mode,method,users_evaluated,overall_NDCG_at_10,overall_Recall_at_10,overall_HitRate_at_10,warm_tail_target_users,warm_tail_NDCG_at_10,warm_tail_Recall_at_10,warm_tail_HitRate_per_profile,warm_tail_users_discovered,warm_tail_discovery_rate_any_repeat,strict_cold_target_users,strict_cold_Recall_at_10
0,3,popularity_biased,ALS fold-in,4527,0.02352,0.03660,0.11012,929,0.0,0.0,0.0,0.0,0.0,76,0.0
3,3,popularity_biased,LightFM Identity centroid,4527,0.02204,0.03406,0.10106,929,0.0,0.0,0.0,0.0,0.0,76,0.0
1,3,popularity_biased,LightFM Content-v1 centroid,4527,0.02180,0.03277,0.09973,929,0.0,0.0,0.0,0.0,0.0,76,0.0
2,3,popularity_biased,LightFM Content-v1 ridge,4527,0.01725,0.02584,0.08295,929,0.0,0.0,0.0,0.0,0.0,76,0.0
4,3,popularity_biased,LightFM Identity ridge,4527,0.01714,0.02585,0.08273,929,0.0,0.0,0.0,0.0,0.0,76,0.0
5,3,uniform,ALS fold-in,4527,0.02516,0.03854,0.11465,929,0.0,0.0,0.0,0.0,0.0,76,0.0
8,3,uniform,LightFM Identity centroid,4527,0.02334,0.03568,0.10581,929,0.0,0.0,0.0,0.0,0.0,76,0.0
6,3,uniform,LightFM Content-v1 centroid,4527,0.02203,0.03350,0.10283,929,0.0,0.0,0.0,0.0,0.0,76,0.0
7,3,uniform,LightFM Content-v1 ridge,4527,0.01743,0.02698,0.08471,929,0.0,0.0,0.0,0.0,0.0,76,0.0
9,3,uniform,LightFM Identity ridge,4527,0.01706,0.02631,0.08460,929,0.0,0.0,0.0,0.0,0.0,76,0.0



B. FOLD-BY-FOLD METRICS — ALS VS CONTENT CENTROID


,fold_id,profile_size,sampling_mode,method,users_evaluated,overall_NDCG_at_10,overall_Recall_at_10,overall_HitRate_at_10,warm_tail_target_users,warm_tail_NDCG_at_10,warm_tail_Recall_at_10,warm_tail_HitRate_per_profile,warm_tail_users_discovered,warm_tail_discovery_rate_any_repeat,strict_cold_target_users,strict_cold_Recall_at_10
0,0,3,popularity_biased,ALS fold-in,906,0.02259,0.03508,0.10044,178,0.0,0.0,0.0,0.0,0.0,16,0.0
40,1,3,popularity_biased,ALS fold-in,906,0.02178,0.03372,0.11258,188,0.0,0.0,0.0,0.0,0.0,16,0.0
80,2,3,popularity_biased,ALS fold-in,905,0.02132,0.03369,0.10663,199,0.0,0.0,0.0,0.0,0.0,18,0.0
120,3,3,popularity_biased,ALS fold-in,905,0.02544,0.03922,0.11160,179,0.0,0.0,0.0,0.0,0.0,14,0.0
160,4,3,popularity_biased,ALS fold-in,905,0.02646,0.04127,0.11934,185,0.0,0.0,0.0,0.0,0.0,12,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36,0,20,uniform,LightFM Content-v1 centroid,746,0.02989,0.04168,0.13539,164,0.0,0.0,0.0,0.0,0.0,14,0.0
76,1,20,uniform,LightFM Content-v1 centroid,746,0.02134,0.02884,0.11059,175,0.0,0.0,0.0,0.0,0.0,16,0.0
116,2,20,uniform,LightFM Content-v1 centroid,771,0.02049,0.02912,0.10895,189,0.0,0.0,0.0,0.0,0.0,15,0.0
156,3,20,uniform,LightFM Content-v1 centroid,756,0.02737,0.03628,0.12566,172,0.0,0.0,0.0,0.0,0.0,13,0.0



C. POOLED WARM-TAIL TARGET RANKS


...

,fold_id,profile_size,sampling_mode,method,rank_observations,median_rank,mean_rank,p90_rank,targets_top10,targets_top100,targets_top500,targets_top1000
0,0,3,popularity_biased,ALS fold-in,562,5590.0,6898.60854,14168.9,0.0,0.00000,0.00178,0.04270
40,1,3,popularity_biased,ALS fold-in,632,5990.5,7495.47943,14670.7,0.0,0.00000,0.00000,0.02373
80,2,3,popularity_biased,ALS fold-in,726,5174.5,6703.91047,14165.0,0.0,0.00000,0.00275,0.03719
120,3,3,popularity_biased,ALS fold-in,622,5067.5,6738.02251,14439.2,0.0,0.00000,0.00000,0.03215
160,4,3,popularity_biased,ALS fold-in,616,5444.5,7000.76948,14545.5,0.0,0.00000,0.00000,0.02110
...,...,...,...,...,...,...,...,...,...,...,...,...
36,0,20,uniform,LightFM Content-v1 centroid,524,3774.5,5024.53435,11531.2,0.0,0.00000,0.01336,0.08969
76,1,20,uniform,LightFM Content-v1 centroid,602,3984.5,4968.50831,10523.5,0.0,0.00831,0.02326,0.07641
116,2,20,uniform,LightFM Content-v1 centroid,698,3456.0,4660.76934,10260.3,0.0,0.00000,0.03152,0.11175
156,3,20,uniform,LightFM Content-v1 centroid,608,3488.0,4629.93586,9561.5,0.0,0.00000,0.01645,0.06743



E. CONTENT CENTROID MINUS ALS — FOLD-LEVEL DIFFERENCES


method,fold_id,profile_size,sampling_mode,metric,content_minus_als
0,0,3,popularity_biased,overall_NDCG_at_10,0.002110
8,1,3,popularity_biased,overall_NDCG_at_10,-0.001272
16,2,3,popularity_biased,overall_NDCG_at_10,-0.003632
24,3,3,popularity_biased,overall_NDCG_at_10,-0.004071
32,4,3,popularity_biased,overall_NDCG_at_10,-0.001699
...,...,...,...,...,...
167,0,20,uniform,warm_tail_discovery_rate_any_repeat,0.000000
175,1,20,uniform,warm_tail_discovery_rate_any_repeat,0.000000
183,2,20,uniform,warm_tail_discovery_rate_any_repeat,0.000000
191,3,20,uniform,warm_tail_discovery_rate_any_repeat,0.000000



F. TRAINING AUDIT


,fold_id,status,als_training_seconds,identity_training_seconds,content_v1_training_seconds,positive_interactions,training_rating_rows,training_users,warm_tail_catalogue_items,warm_tail_validation_pairs,sampled_profiles,total_fold_seconds
0,0,reused_smoke_models,56.42,NaN,NaN,1248910,2533620,18055,10469,281,13804,NaN
1,1,trained,41.60,21.18,31.86,1244919,2530213,18055,10461,316,13812,344.38
2,2,trained,39.94,19.95,34.78,1238195,2514835,18056,10430,363,13876,340.11
3,3,trained,39.32,22.49,36.78,1245566,2528523,18056,10491,311,13808,366.47
4,4,trained,39.41,22.43,33.88,1244006,2516448,18056,10384,308,13872,334.17


In [15]:

# Fold-wise consistency: Content-v1 centroid vs ALS

idx = ["fold_id", "profile_size", "sampling_mode"]

top500 = fold_ranks.pivot(
    index=idx,
    columns="method",
    values="targets_top500",
)

top1000 = fold_ranks.pivot(
    index=idx,
    columns="method",
    values="targets_top1000",
)

median_rank = fold_ranks.pivot(
    index=idx,
    columns="method",
    values="median_rank",
)

ndcg = fold_metrics.pivot(
    index=idx,
    columns="method",
    values="overall_NDCG_at_10",
)

als = "ALS fold-in"
content = "LightFM Content-v1 centroid"

fold_consistency = pd.DataFrame({
    "content_wins_top500": (
        top500[content] > top500[als]
    ),
    "content_wins_top1000": (
        top1000[content] > top1000[als]
    ),
    "content_has_better_median_rank": (
        median_rank[content] < median_rank[als]
    ),
    "content_wins_overall_ndcg": (
        ndcg[content] > ndcg[als]
    ),
}).reset_index()

consistency_summary = (
    fold_consistency
    .groupby(["profile_size", "sampling_mode"], as_index=False)
    .agg(
        folds_better_top500=("content_wins_top500", "sum"),
        folds_better_top1000=("content_wins_top1000", "sum"),
        folds_better_median_rank=(
            "content_has_better_median_rank", "sum"
        ),
        folds_better_overall_ndcg=("content_wins_overall_ndcg", "sum"),
        total_folds=("fold_id", "nunique"),
    )
)

display(consistency_summary)


,profile_size,sampling_mode,folds_better_top500,folds_better_top1000,folds_better_median_rank,folds_better_overall_ndcg,total_folds
0,3,popularity_biased,5,5,2,1,5
1,3,uniform,5,5,4,1,5
2,5,popularity_biased,4,5,1,0,5
3,5,uniform,5,5,5,0,5
4,10,popularity_biased,4,4,1,0,5
5,10,uniform,5,5,4,0,5
6,20,popularity_biased,5,5,0,0,5
7,20,uniform,5,5,1,0,5


In [16]:

# ============================================================
# CONTENT-TO-ALS FACTOR IMPUTATION — FOLD 0
# ============================================================

from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import numpy as np
import pandas as pd
import time

FACTOR_RIDGE_ALPHA = 10.0
FACTOR_TEST_SIZE = 0.20
FACTOR_RANDOM_STATE = SEED + 501

# ------------------------------------------------------------
# 1. Prepare metadata and ALS factor targets
# ------------------------------------------------------------

assert X_v1.shape == (n_items, 31)
assert Q_als_smoke.shape == (n_items, ALS_FACTORS)

# Convert the 31-column sparse feature matrix to dense.
# 17,241 x 31 is small enough for memory.
X_factor = X_v1.toarray().astype(np.float32)

assert X_factor.shape == (n_items, 31)
assert np.isfinite(X_factor).all()

# Items without metadata cannot be used by this mapper.
has_metadata = np.asarray(
    X_v1.getnnz(axis=1) > 0
).ravel()

# Train the mapper only on well-supported head items.
head_mapper_pool = np.flatnonzero(
    smoke_fold["head_mask"] & has_metadata
)

# Evaluate factor prediction on held-out head items.
mapper_train_ids, mapper_valid_ids = train_test_split(
    head_mapper_pool,
    test_size=FACTOR_TEST_SIZE,
    random_state=FACTOR_RANDOM_STATE,
)

print("Head items with metadata:", len(head_mapper_pool))
print("Mapper training items:", len(mapper_train_ids))
print("Mapper validation items:", len(mapper_valid_ids))

assert len(mapper_train_ids) > X_factor.shape[1]
assert len(mapper_valid_ids) > 0


# ------------------------------------------------------------
# 2. Fit Ridge on the training head items
# ------------------------------------------------------------

factor_mapper_validation = Ridge(
    alpha=FACTOR_RIDGE_ALPHA,
    fit_intercept=True,
    solver="lsqr",
)

start = time.perf_counter()

factor_mapper_validation.fit(
    X_factor[mapper_train_ids],
    Q_als_smoke[mapper_train_ids],
)

validation_training_seconds = time.perf_counter() - start

# Predict ALS item factors for held-out head items.
Q_valid_true = Q_als_smoke[mapper_valid_ids]
Q_valid_pred = factor_mapper_validation.predict(
    X_factor[mapper_valid_ids]
).astype(np.float32)

assert Q_valid_pred.shape == Q_valid_true.shape
assert np.isfinite(Q_valid_pred).all()


# ------------------------------------------------------------
# 3. Evaluate factor reconstruction quality
# ------------------------------------------------------------

def factor_prediction_diagnostics(y_true, y_pred):

    residual = y_true - y_pred

    rmse = float(
        np.sqrt(np.mean(residual ** 2))
    )

    target_std = float(np.std(y_true))

    normalized_rmse = (
        rmse / target_std
        if target_std > 0 else np.nan
    )

    # Variance-weighted R² across all 64 factor dimensions.
    weighted_r2 = float(
        r2_score(
            y_true,
            y_pred,
            multioutput="variance_weighted",
        )
    )

    # Per-item cosine similarity.
    true_norms = np.linalg.norm(y_true, axis=1)
    pred_norms = np.linalg.norm(y_pred, axis=1)

    denominator = true_norms * pred_norms

    cosine = np.divide(
        np.sum(y_true * y_pred, axis=1),
        denominator,
        out=np.zeros_like(true_norms, dtype=np.float64),
        where=denominator > 0,
    )

    norm_ratio = np.divide(
        pred_norms,
        true_norms,
        out=np.zeros_like(true_norms, dtype=np.float64),
        where=true_norms > 0,
    )

    # Baseline 1: predicting zero for every factor.
    zero_rmse = float(
        np.sqrt(np.mean(y_true ** 2))
    )

    # Baseline 2: predicting the training-set mean factor.
    mean_vector = Q_als_smoke[mapper_train_ids].mean(axis=0)
    mean_pred = np.broadcast_to(
        mean_vector,
        y_true.shape,
    )

    mean_rmse = float(
        np.sqrt(np.mean((y_true - mean_pred) ** 2))
    )

    return {
        "RMSE": rmse,
        "normalized_RMSE": normalized_rmse,
        "variance_weighted_R2": weighted_r2,
        "mean_cosine_similarity": float(np.mean(cosine)),
        "median_cosine_similarity": float(np.median(cosine)),
        "median_predicted_to_true_norm_ratio": float(
            np.median(norm_ratio)
        ),
        "zero_baseline_RMSE": zero_rmse,
        "mean_baseline_RMSE": mean_rmse,
    }


validation_diagnostics = factor_prediction_diagnostics(
    Q_valid_true,
    Q_valid_pred,
)

print("\n=== HELD-OUT HEAD ITEM FACTOR VALIDATION ===")
display(
    pd.Series(validation_diagnostics, name="value").to_frame()
)

print(
    "\nValidation mapper training time:",
    f"{validation_training_seconds:.2f} seconds"
)


# ------------------------------------------------------------
# 4. Refit on all head items
# ------------------------------------------------------------

factor_mapper_fold0 = Ridge(
    alpha=FACTOR_RIDGE_ALPHA,
    fit_intercept=True,
    solver="lsqr",
)

start = time.perf_counter()

factor_mapper_fold0.fit(
    X_factor[head_mapper_pool],
    Q_als_smoke[head_mapper_pool],
)

final_mapper_training_seconds = time.perf_counter() - start


# ------------------------------------------------------------
# 5. Predict factors for warm-tail items
# ------------------------------------------------------------

warm_tail_with_metadata = np.flatnonzero(
    smoke_fold["warm_tail_mask"] & has_metadata
)

warm_tail_without_metadata = np.flatnonzero(
    smoke_fold["warm_tail_mask"] & ~has_metadata
)

Q_warm_tail_pred = factor_mapper_fold0.predict(
    X_factor[warm_tail_with_metadata]
).astype(np.float32)

assert Q_warm_tail_pred.shape == (
    len(warm_tail_with_metadata),
    ALS_FACTORS,
)

assert np.isfinite(Q_warm_tail_pred).all()

# Important: retain the original ALS factors separately.
# Only replace warm-tail candidate factors in this copy.
Q_als_imputed_fold0 = Q_als_smoke.copy()

Q_als_imputed_fold0[warm_tail_with_metadata] = (
    Q_warm_tail_pred
)

# Items without metadata and strict-cold items remain unchanged.
assert np.array_equal(
    Q_als_imputed_fold0[~smoke_fold["warm_tail_mask"]],
    Q_als_smoke[~smoke_fold["warm_tail_mask"]],
)

assert np.isfinite(Q_als_imputed_fold0).all()


# ------------------------------------------------------------
# 6. Compare original and imputed warm-tail factor norms
# ------------------------------------------------------------

original_tail_norms = np.linalg.norm(
    Q_als_smoke[warm_tail_with_metadata],
    axis=1,
)

imputed_tail_norms = np.linalg.norm(
    Q_warm_tail_pred,
    axis=1,
)

factor_norm_summary = pd.DataFrame({
    "original_ALS_tail_norm": original_tail_norms,
    "imputed_tail_norm": imputed_tail_norms,
})

print("\n=== FACTOR NORM COMPARISON ===")

display(
    factor_norm_summary.describe().round(5)
)

print("\n=== IMPUTATION COVERAGE ===")

imputation_summary = pd.Series({
    "warm_tail_items_total": int(
        smoke_fold["warm_tail_mask"].sum()
    ),
    "warm_tail_items_with_metadata": len(
        warm_tail_with_metadata
    ),
    "warm_tail_items_without_metadata": len(
        warm_tail_without_metadata
    ),
    "warm_tail_imputation_coverage": (
        len(warm_tail_with_metadata)
        / max(int(smoke_fold["warm_tail_mask"].sum()), 1)
    ),
    "head_items_used_for_mapper": len(head_mapper_pool),
    "mapper_alpha": FACTOR_RIDGE_ALPHA,
    "mapper_training_seconds": final_mapper_training_seconds,
})

display(imputation_summary.to_frame(name="value"))

print("\nFold-0 content-to-ALS mapping completed.")
print("Original ALS factors remain unchanged:", 
      np.array_equal(Q_als_smoke, als_smoke["Q"]))


Head items with metadata: 5166
Mapper training items: 4132
Mapper validation items: 1034

=== HELD-OUT HEAD ITEM FACTOR VALIDATION ===


,value
RMSE,0.117449
normalized_RMSE,0.883640
variance_weighted_R2,0.150010
mean_cosine_similarity,0.583629
median_cosine_similarity,0.590146
median_predicted_to_true_norm_ratio,0.809284
zero_baseline_RMSE,0.132982
mean_baseline_RMSE,0.127460



Validation mapper training time: 0.12 seconds

=== FACTOR NORM COMPARISON ===


,original_ALS_tail_norm,imputed_tail_norm
count,10458.00000,10458.00000
mean,0.11545,0.46329
std,0.06995,0.05561
min,0.00121,0.28661
25%,0.06011,0.42495
50%,0.10050,0.45518
75%,0.16020,0.49708
max,0.37148,0.74301



=== IMPUTATION COVERAGE ===


,value
warm_tail_items_total,10469.000000
warm_tail_items_with_metadata,10458.000000
warm_tail_items_without_metadata,11.000000
warm_tail_imputation_coverage,0.998949
head_items_used_for_mapper,5166.000000
mapper_alpha,10.000000
mapper_training_seconds,0.105196



Fold-0 content-to-ALS mapping completed.
Original ALS factors remain unchanged: True


In [17]:

# ============================================================
# FOLD-0 RANKING EVALUATION: ORIGINAL ALS vs IMPUTED ALS
# ============================================================

assert Q_als_imputed_fold0.shape == Q_als_smoke.shape
assert np.isfinite(Q_als_imputed_fold0).all()
assert np.array_equal(
    Q_als_smoke,
    als_smoke["Q"],
), "Original ALS factors have been modified."

assert not np.array_equal(
    Q_als_imputed_fold0,
    Q_als_smoke,
), "No item factors were changed by imputation."

FACTOR_EVAL_METHODS = [
    "ALS original",
    "ALS content-imputed",
]

factor_eval_rows = []
factor_tail_rank_rows = []

item_ids = np.arange(n_items, dtype=np.int64)

# Catalogue segments for inspecting recommendations.
factor_item_segment = np.full(
    n_items,
    "strict-cold",
    dtype=object,
)
factor_item_segment[smoke_fold["warm_tail_mask"]] = "warm-tail"
factor_item_segment[smoke_fold["head_mask"]] = "head"

start = time.perf_counter()

print(
    "Profiles to evaluate:",
    f"{len(smoke_fold['support_samples']):,}"
)

for profile_number, (key, profile_items) in enumerate(
    smoke_fold["support_samples"].items(),
    start=1,
):

    user_id, profile_size, sampling_mode, repeat_id = key

    user_id = int(user_id)
    profile_items = np.asarray(profile_items, dtype=np.int64)

    overall_targets = smoke_fold["overall_relevant"].get(
        user_id, set()
    )
    warm_targets = smoke_fold["warm_tail_relevant"].get(
        user_id, set()
    )
    cold_targets = smoke_fold["strict_cold_relevant"].get(
        user_id, set()
    )

    assert overall_targets
    assert not (set(profile_items) & overall_targets)

    # IMPORTANT:
    # Infer the anonymous user using ORIGINAL ALS item factors.
    # This produces exactly the same user vector for both candidate scorers.
    user_vector = fold_in_als_profile(
        profile_items,
        Q_als_smoke,
    )

    # Baseline candidate scores.
    scores_original = (
        Q_als_smoke @ user_vector
    ).astype(np.float64)

    # Imputed candidate scores.
    scores_imputed = (
        Q_als_imputed_fold0 @ user_vector
    ).astype(np.float64)

    score_methods = {
        "ALS original": scores_original,
        "ALS content-imputed": scores_imputed,
    }

    for method_name, raw_scores in score_methods.items():

        scores = raw_scores.copy()

        # Exclude selected profile movies.
        scores[profile_items] = -np.inf

        assert np.isfinite(scores[np.isfinite(scores)]).all()

        # Deterministic full-catalogue ranking.
        rank_order = np.lexsort((item_ids, -scores))
        top10 = rank_order[:10]

        assert len(top10) == 10
        assert not (set(top10) & set(profile_items))

        rank_by_item = np.empty(n_items, dtype=np.int32)
        rank_by_item[rank_order] = np.arange(
            1,
            n_items + 1,
            dtype=np.int32,
        )

        overall_metrics = _cv_ranking_metrics(
            top10,
            overall_targets,
        )

        warm_metrics = _cv_ranking_metrics(
            top10,
            warm_targets,
        )

        cold_metrics = _cv_ranking_metrics(
            top10,
            cold_targets,
        )

        top10_segments = [
            factor_item_segment[item]
            for item in top10
        ]

        factor_eval_rows.append({
            "fold_id": 0,
            "user_id": user_id,
            "profile_size": int(profile_size),
            "sampling_mode": sampling_mode,
            "repeat_id": int(repeat_id),
            "method": method_name,

            "n_overall_targets": len(overall_targets),
            "n_warm_tail_targets": len(warm_targets),
            "n_strict_cold_targets": len(cold_targets),

            "overall_ndcg": overall_metrics["ndcg"],
            "overall_recall": overall_metrics["recall"],
            "overall_hit": overall_metrics["hit"],

            "warm_tail_ndcg": warm_metrics["ndcg"],
            "warm_tail_recall": warm_metrics["recall"],
            "warm_tail_hit": warm_metrics["hit"],

            "strict_cold_recall": cold_metrics["recall"],

            "warm_tail_items_in_top10": top10_segments.count(
                "warm-tail"
            ),
            "head_items_in_top10": top10_segments.count("head"),
            "strict_cold_items_in_top10": top10_segments.count(
                "strict-cold"
            ),

            "top10_item_indices": top10.tolist(),
        })

        # Exact ranks for all warm-tail validation targets.
        for item_id in warm_targets:

            item_id = int(item_id)
            rank = int(rank_by_item[item_id])

            factor_tail_rank_rows.append({
                "fold_id": 0,
                "user_id": user_id,
                "profile_size": int(profile_size),
                "sampling_mode": sampling_mode,
                "repeat_id": int(repeat_id),
                "method": method_name,
                "item_index": item_id,
                "rank": rank,
                "rank_percentile": (
                    100.0 * (rank - 1) / (n_items - 1)
                ),
            })

    if profile_number % 2000 == 0:
        print(
            f"Processed {profile_number:,}/"
            f"{len(smoke_fold['support_samples']):,} profiles"
        )


# ------------------------------------------------------------
# Build result dataframes
# ------------------------------------------------------------

factor_imputation_evaluation = pd.DataFrame(
    factor_eval_rows
)

factor_imputation_tail_ranks = pd.DataFrame(
    factor_tail_rank_rows
)

assert not factor_imputation_evaluation.empty
assert not factor_imputation_tail_ranks.empty

assert set(
    factor_imputation_evaluation["method"].unique()
) == set(FACTOR_EVAL_METHODS)

print("\nEvaluation completed.")
print(
    "Profile-method evaluations:",
    f"{len(factor_imputation_evaluation):,}"
)
print(
    "Warm-tail target rank observations:",
    f"{len(factor_imputation_tail_ranks):,}"
)
print(
    "Evaluation time:",
    f"{(time.perf_counter() - start):.2f} seconds"
)


# ------------------------------------------------------------
# Average repeats within each user
# ------------------------------------------------------------

factor_user_level = (
    factor_imputation_evaluation
    .groupby(
        [
            "fold_id",
            "user_id",
            "profile_size",
            "sampling_mode",
            "method",
        ],
        as_index=False,
    )
    .agg(
        n_overall_targets=("n_overall_targets", "max"),
        n_warm_tail_targets=("n_warm_tail_targets", "max"),
        n_strict_cold_targets=("n_strict_cold_targets", "max"),

        overall_ndcg=("overall_ndcg", "mean"),
        overall_recall=("overall_recall", "mean"),
        overall_hit_rate=("overall_hit", "mean"),

        warm_tail_ndcg=("warm_tail_ndcg", "mean"),
        warm_tail_recall=("warm_tail_recall", "mean"),
        warm_tail_hit_any_repeat=("warm_tail_hit", "max"),

        strict_cold_recall=("strict_cold_recall", "mean"),

        mean_warm_tail_items_in_top10=(
            "warm_tail_items_in_top10", "mean"
        ),
    )
)

factor_user_level["has_warm_tail_target"] = (
    factor_user_level["n_warm_tail_targets"] > 0
)

factor_user_level["warm_tail_discovery"] = np.where(
    factor_user_level["has_warm_tail_target"],
    factor_user_level["warm_tail_hit_any_repeat"],
    np.nan,
)


# ------------------------------------------------------------
# Pooled fold-0 summary
# ------------------------------------------------------------

factor_imputation_summary = (
    factor_user_level
    .groupby(
        ["profile_size", "sampling_mode", "method"],
        as_index=False,
    )
    .agg(
        users_evaluated=("user_id", "nunique"),

        overall_NDCG_at_10=("overall_ndcg", "mean"),
        overall_Recall_at_10=("overall_recall", "mean"),
        overall_HitRate_at_10=("overall_hit_rate", "mean"),

        warm_tail_target_users=("has_warm_tail_target", "sum"),
        warm_tail_Recall_at_10=("warm_tail_recall", "mean"),
        warm_tail_HitRate_per_profile=(
            "warm_tail_hit_any_repeat", "mean"
        ),
        warm_tail_users_discovered=(
            "warm_tail_discovery", "sum"
        ),
        warm_tail_discovery_rate_any_repeat=(
            "warm_tail_discovery", "mean"
        ),

        mean_warm_tail_items_in_top10=(
            "mean_warm_tail_items_in_top10", "mean"
        ),

        strict_cold_Recall_at_10=("strict_cold_recall", "mean"),
    )
)


# ------------------------------------------------------------
# Warm-tail target rank summary
# ------------------------------------------------------------

factor_imputation_rank_summary = (
    factor_imputation_tail_ranks
    .groupby(
        ["profile_size", "sampling_mode", "method"],
        as_index=False,
    )
    .agg(
        rank_observations=("rank", "size"),
        median_warm_tail_rank=("rank", "median"),
        mean_warm_tail_rank=("rank", "mean"),
        p90_warm_tail_rank=(
            "rank",
            lambda x: x.quantile(0.90),
        ),
        pct_targets_top10=("rank", lambda x: (x <= 10).mean()),
        pct_targets_top100=("rank", lambda x: (x <= 100).mean()),
        pct_targets_top500=("rank", lambda x: (x <= 500).mean()),
        pct_targets_top1000=("rank", lambda x: (x <= 1000).mean()),
    )
)


# ------------------------------------------------------------
# Direct deltas: content-imputed minus original ALS
# ------------------------------------------------------------

comparison_metrics = [
    "overall_NDCG_at_10",
    "overall_Recall_at_10",
    "warm_tail_Recall_at_10",
    "warm_tail_HitRate_per_profile",
    "warm_tail_discovery_rate_any_repeat",
    "mean_warm_tail_items_in_top10",
]

factor_delta_rows = []

for metric in comparison_metrics:

    wide = factor_imputation_summary.pivot(
        index=["profile_size", "sampling_mode"],
        columns="method",
        values=metric,
    )

    delta = (
        wide["ALS content-imputed"]
        - wide["ALS original"]
    )

    for (profile_size, sampling_mode), value in delta.items():

        factor_delta_rows.append({
            "profile_size": profile_size,
            "sampling_mode": sampling_mode,
            "metric": metric,
            "imputed_minus_original": value,
        })

factor_imputation_deltas = pd.DataFrame(factor_delta_rows)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\nA. ORIGINAL ALS VS CONTENT-IMPUTED ALS")
display(
    factor_imputation_summary
    .sort_values(
        ["profile_size", "sampling_mode", "method"]
    )
    .round(5)
)

print("\nB. WARM-TAIL TARGET RANK DISTRIBUTIONS")
display(
    factor_imputation_rank_summary
    .sort_values(
        ["profile_size", "sampling_mode", "method"]
    )
    .round(5)
)

print("\nC. IMPUTED MINUS ORIGINAL ALS")
display(
    factor_imputation_deltas
    .sort_values(
        ["profile_size", "sampling_mode", "metric"]
    )
    .round(6)
)

print("\nD. SAMPLE TOP-10 RECOMMENDATIONS")
display(
    factor_imputation_evaluation[
        [
            "user_id",
            "profile_size",
            "sampling_mode",
            "repeat_id",
            "method",
            "top10_item_indices",
            "warm_tail_items_in_top10",
        ]
    ].head(20)
)

print("\nFold-0 factor-imputation ranking evaluation complete.")
print("Original ALS model and previous experiment tables remain unchanged.")


Profiles to evaluate: 13,804
Processed 2,000/13,804 profiles
Processed 4,000/13,804 profiles
Processed 6,000/13,804 profiles
Processed 8,000/13,804 profiles
Processed 10,000/13,804 profiles
Processed 12,000/13,804 profiles

Evaluation completed.
Profile-method evaluations: 27,608
Warm-tail target rank observations: 8,840
Evaluation time: 120.76 seconds

A. ORIGINAL ALS VS CONTENT-IMPUTED ALS


,profile_size,sampling_mode,method,users_evaluated,overall_NDCG_at_10,overall_Recall_at_10,overall_HitRate_at_10,warm_tail_target_users,warm_tail_Recall_at_10,warm_tail_HitRate_per_profile,warm_tail_users_discovered,warm_tail_discovery_rate_any_repeat,mean_warm_tail_items_in_top10,strict_cold_Recall_at_10
0,3,popularity_biased,ALS content-imputed,906,0.02259,0.03508,0.10044,178,0.0,0.0,0.0,0.0,0.0,0.0
1,3,popularity_biased,ALS original,906,0.02259,0.03508,0.10044,178,0.0,0.0,0.0,0.0,0.0,0.0
2,3,uniform,ALS content-imputed,906,0.02636,0.04186,0.11479,178,0.0,0.0,0.0,0.0,0.0,0.0
3,3,uniform,ALS original,906,0.02636,0.04186,0.11479,178,0.0,0.0,0.0,0.0,0.0,0.0
4,5,popularity_biased,ALS content-imputed,906,0.03154,0.04756,0.13411,178,0.0,0.0,0.0,0.0,0.0,0.0
5,5,popularity_biased,ALS original,906,0.03154,0.04756,0.13411,178,0.0,0.0,0.0,0.0,0.0,0.0
6,5,uniform,ALS content-imputed,906,0.02753,0.04294,0.12914,178,0.0,0.0,0.0,0.0,0.0,0.0
7,5,uniform,ALS original,906,0.02753,0.04294,0.12914,178,0.0,0.0,0.0,0.0,0.0,0.0
8,10,popularity_biased,ALS content-imputed,893,0.03373,0.05027,0.14670,178,0.0,0.0,0.0,0.0,0.0,0.0
9,10,popularity_biased,ALS original,893,0.03373,0.05027,0.14670,178,0.0,0.0,0.0,0.0,0.0,0.0



B. WARM-TAIL TARGET RANK DISTRIBUTIONS


,profile_size,sampling_mode,method,rank_observations,median_warm_tail_rank,mean_warm_tail_rank,p90_warm_tail_rank,pct_targets_top10,pct_targets_top100,pct_targets_top500,pct_targets_top1000
0,3,popularity_biased,ALS content-imputed,562,4474.5,5934.31495,13627.5,0.0,0.0,0.04270,0.11744
1,3,popularity_biased,ALS original,562,5590.0,6898.60854,14168.9,0.0,0.0,0.00178,0.04270
2,3,uniform,ALS content-imputed,562,3672.5,4846.70463,11620.5,0.0,0.0,0.05516,0.15302
3,3,uniform,ALS original,562,4845.5,6189.79715,13790.3,0.0,0.0,0.00356,0.06228
4,5,popularity_biased,ALS content-imputed,562,3928.0,5193.34342,12505.0,0.0,0.0,0.04626,0.12811
5,5,popularity_biased,ALS original,562,5031.0,6418.97687,13479.0,0.0,0.0,0.00000,0.04270
6,5,uniform,ALS content-imputed,562,3420.0,4821.02847,11078.5,0.0,0.0,0.04270,0.16014
7,5,uniform,ALS original,562,4295.0,5659.43950,13056.8,0.0,0.0,0.00178,0.04270
8,10,popularity_biased,ALS content-imputed,562,3367.0,4728.71530,11620.6,0.0,0.0,0.05338,0.14947
9,10,popularity_biased,ALS original,562,4236.0,5954.29715,13653.7,0.0,0.0,0.00000,0.03915



C. IMPUTED MINUS ORIGINAL ALS


,profile_size,sampling_mode,metric,imputed_minus_original
40,3,popularity_biased,mean_warm_tail_items_in_top10,0.0
0,3,popularity_biased,overall_NDCG_at_10,0.0
8,3,popularity_biased,overall_Recall_at_10,0.0
24,3,popularity_biased,warm_tail_HitRate_per_profile,0.0
16,3,popularity_biased,warm_tail_Recall_at_10,0.0
32,3,popularity_biased,warm_tail_discovery_rate_any_repeat,0.0
41,3,uniform,mean_warm_tail_items_in_top10,0.0
1,3,uniform,overall_NDCG_at_10,0.0
9,3,uniform,overall_Recall_at_10,0.0
25,3,uniform,warm_tail_HitRate_per_profile,0.0



D. SAMPLE TOP-10 RECOMMENDATIONS


,user_id,profile_size,sampling_mode,repeat_id,method,top10_item_indices,warm_tail_items_in_top10
0,148,3,uniform,0,ALS original,"[12614, 10930, 10287, 13574, 9698, 12088, 1295...",0
1,148,3,uniform,0,ALS content-imputed,"[12614, 10930, 10287, 13574, 9698, 12088, 1295...",0
2,148,3,uniform,1,ALS original,"[5650, 3868, 10287, 7261, 0, 9146, 10903, 2817...",0
3,148,3,uniform,1,ALS content-imputed,"[5650, 3868, 10287, 7261, 0, 9146, 10903, 2817...",0
4,148,3,popularity_biased,0,ALS original,"[563, 305, 342, 2675, 12088, 10930, 12614, 46,...",0
5,148,3,popularity_biased,0,ALS content-imputed,"[563, 305, 342, 2675, 12088, 10930, 12614, 46,...",0
6,148,3,popularity_biased,1,ALS original,"[4396, 7261, 3868, 9812, 10287, 6062, 0, 6519,...",0
7,148,3,popularity_biased,1,ALS content-imputed,"[4396, 7261, 3868, 9812, 10287, 6062, 0, 6519,...",0
8,148,5,uniform,0,ALS original,"[563, 6880, 6062, 305, 9359, 5187, 6519, 4405,...",0
9,148,5,uniform,0,ALS content-imputed,"[563, 6880, 6062, 305, 9359, 5187, 6519, 4405,...",0



Fold-0 factor-imputation ranking evaluation complete.
Original ALS model and previous experiment tables remain unchanged.


In [18]:

# ============================================================
# CONTENT-TO-ALS IMPUTATION — FIVE-FOLD DEVELOPMENT EVALUATION
# Fold 0 is reused. Folds 1-4 fit ALS + their own content mapper.
# ============================================================

from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import gc
import time

FACTOR_RIDGE_ALPHA = 10.0

# Preserve the already-completed fold-0 results.
factor_cv_eval_parts = [
    factor_imputation_evaluation.copy()
]
factor_cv_rank_parts = [
    factor_imputation_tail_ranks.copy()
]

factor_cv_mapper_diagnostics = []
factor_cv_fit_audit = []

item_ids = np.arange(n_items, dtype=np.int64)

# X_factor and has_metadata were created in the fold-0 mapper cell.
assert X_factor.shape == (n_items, 31)
assert has_metadata.shape == (n_items,)


def _factor_mapper_diagnostics(y_true, y_pred, y_train):
    """Evaluate factor reconstruction on held-out head items."""

    residual = y_true - y_pred
    rmse = float(np.sqrt(np.mean(residual ** 2)))

    true_norms = np.linalg.norm(y_true, axis=1)
    pred_norms = np.linalg.norm(y_pred, axis=1)

    denominator = true_norms * pred_norms

    cosine = np.divide(
        np.sum(y_true * y_pred, axis=1),
        denominator,
        out=np.zeros_like(true_norms, dtype=np.float64),
        where=denominator > 0,
    )

    norm_ratio = np.divide(
        pred_norms,
        true_norms,
        out=np.zeros_like(true_norms, dtype=np.float64),
        where=true_norms > 0,
    )

    mean_vector = y_train.mean(axis=0)
    mean_prediction = np.broadcast_to(mean_vector, y_true.shape)

    return {
        "RMSE": rmse,
        "normalized_RMSE": float(
            rmse / np.std(y_true)
        ),
        "variance_weighted_R2": float(
            r2_score(
                y_true,
                y_pred,
                multioutput="variance_weighted",
            )
        ),
        "mean_cosine_similarity": float(np.mean(cosine)),
        "median_cosine_similarity": float(np.median(cosine)),
        "median_predicted_to_true_norm_ratio": float(
            np.median(norm_ratio)
        ),
        "zero_baseline_RMSE": float(
            np.sqrt(np.mean(y_true ** 2))
        ),
        "mean_baseline_RMSE": float(
            np.sqrt(np.mean((y_true - mean_prediction) ** 2))
        ),
    }


# ------------------------------------------------------------
# FOLDS 1-4
# ------------------------------------------------------------

for fold_id in range(1, N_FOLDS):

    print("\n" + "=" * 75)
    print(f"CONTENT-TO-ALS — FOLD {fold_id}")
    print("=" * 75)

    fold_start = time.perf_counter()

    # Build exactly the same leakage-safe fold context.
    fold_ctx = _build_fold_context(
        fold_id,
        support_repeats=CV_SUPPORT_REPEATS,
    )

    heldout_set = set(map(int, fold_ctx["heldout_users"]))
    pilot_set = set(map(int, pilot_confirmation_users))
    excluded_set = set(map(int, fold_ctx["excluded_users"]))

    assert heldout_set.isdisjoint(pilot_set)
    assert excluded_set == heldout_set | pilot_set

    assert not fold_ctx["fold_train"]["u"].isin(
        excluded_set
    ).any()

    # --------------------------------------------------------
    # 1. Fit fold-specific ALS
    # --------------------------------------------------------

    print(f"Fitting ALS for fold {fold_id}...")

    als_fit = fit_fold_implicit_als(
        fold_ctx["fold_train"],
        k=ALS_FACTORS,
        alpha=ALS_ALPHA,
        lam=ALS_LAMBDA,
        n_iter=ALS_ITERATIONS,
        seed=SEED,
    )

    Q_als = als_fit["Q"]

    assert Q_als.shape == (n_items, ALS_FACTORS)
    assert np.isfinite(Q_als).all()

    # --------------------------------------------------------
    # 2. Prepare head and warm-tail item groups
    # --------------------------------------------------------

    head_mapper_pool = np.flatnonzero(
        fold_ctx["head_mask"] & has_metadata
    )

    warm_tail_with_metadata = np.flatnonzero(
        fold_ctx["warm_tail_mask"] & has_metadata
    )

    warm_tail_without_metadata = np.flatnonzero(
        fold_ctx["warm_tail_mask"] & ~has_metadata
    )

    assert len(head_mapper_pool) > X_factor.shape[1]
    assert len(warm_tail_with_metadata) > 0

    # --------------------------------------------------------
    # 3. Held-out head-item mapper diagnostic
    # --------------------------------------------------------

    mapper_train_ids, mapper_valid_ids = train_test_split(
        head_mapper_pool,
        test_size=FACTOR_TEST_SIZE,
        random_state=FACTOR_RANDOM_STATE + fold_id,
    )

    mapper_validation = Ridge(
        alpha=FACTOR_RIDGE_ALPHA,
        fit_intercept=True,
        solver="lsqr",
    )

    mapper_validation.fit(
        X_factor[mapper_train_ids],
        Q_als[mapper_train_ids],
    )

    Q_valid_pred = mapper_validation.predict(
        X_factor[mapper_valid_ids]
    ).astype(np.float32)

    diagnostics = _factor_mapper_diagnostics(
        Q_als[mapper_valid_ids],
        Q_valid_pred,
        Q_als[mapper_train_ids],
    )

    diagnostics.update({
        "fold_id": fold_id,
        "head_items_with_metadata": len(head_mapper_pool),
        "mapper_train_items": len(mapper_train_ids),
        "mapper_validation_items": len(mapper_valid_ids),
    })

    factor_cv_mapper_diagnostics.append(diagnostics)

    print("\nHeld-out head-item factor diagnostics:")
    display(pd.Series(diagnostics).to_frame("value").round(5))

    # --------------------------------------------------------
    # 4. Refit mapper on all head items in this fold
    # --------------------------------------------------------

    factor_mapper = Ridge(
        alpha=FACTOR_RIDGE_ALPHA,
        fit_intercept=True,
        solver="lsqr",
    )

    factor_mapper.fit(
        X_factor[head_mapper_pool],
        Q_als[head_mapper_pool],
    )

    # --------------------------------------------------------
    # 5. Impute only warm-tail items with metadata
    # --------------------------------------------------------

    Q_warm_tail_pred = factor_mapper.predict(
        X_factor[warm_tail_with_metadata]
    ).astype(np.float32)

    assert np.isfinite(Q_warm_tail_pred).all()

    Q_als_imputed = Q_als.copy()

    Q_als_imputed[warm_tail_with_metadata] = (
        Q_warm_tail_pred
    )

    # All non-imputed items must remain exactly unchanged.
    unchanged_mask = np.ones(n_items, dtype=bool)
    unchanged_mask[warm_tail_with_metadata] = False

    assert np.array_equal(
        Q_als_imputed[unchanged_mask],
        Q_als[unchanged_mask],
    )

    assert np.isfinite(Q_als_imputed).all()

    # --------------------------------------------------------
    # 6. Evaluate original vs imputed item factors
    # --------------------------------------------------------

    eval_rows = []
    rank_rows = []

    factor_item_segment = np.full(
        n_items,
        "strict-cold",
        dtype=object,
    )
    factor_item_segment[fold_ctx["warm_tail_mask"]] = "warm-tail"
    factor_item_segment[fold_ctx["head_mask"]] = "head"

    support_samples = fold_ctx["support_samples"]

    print(
        f"\nEvaluating {len(support_samples):,} profiles "
        f"for fold {fold_id}..."
    )

    for profile_number, (key, profile_items) in enumerate(
        support_samples.items(),
        start=1,
    ):

        user_id, profile_size, sampling_mode, repeat_id = key

        user_id = int(user_id)
        profile_items = np.asarray(
            profile_items,
            dtype=np.int64,
        )

        overall_targets = fold_ctx["overall_relevant"].get(
            user_id, set()
        )
        warm_targets = fold_ctx["warm_tail_relevant"].get(
            user_id, set()
        )
        cold_targets = fold_ctx["strict_cold_relevant"].get(
            user_id, set()
        )

        assert overall_targets
        assert not (set(profile_items) & overall_targets)

        # Infer once from ORIGINAL ALS factors.
        # The user representation is identical for both scorers.
        user_vector = fold_in_als_profile(
            profile_items,
            Q_als,
        )

        scores_original = (
            Q_als @ user_vector
        ).astype(np.float64)

        scores_imputed = (
            Q_als_imputed @ user_vector
        ).astype(np.float64)

        score_methods = {
            "ALS original": scores_original,
            "ALS content-imputed": scores_imputed,
        }

        for method_name, raw_scores in score_methods.items():

            scores = raw_scores.copy()
            scores[profile_items] = -np.inf

            rank_order = np.lexsort((item_ids, -scores))
            top10 = rank_order[:10]

            assert len(top10) == 10
            assert not (set(top10) & set(profile_items))

            rank_by_item = np.empty(
                n_items,
                dtype=np.int32,
            )
            rank_by_item[rank_order] = np.arange(
                1,
                n_items + 1,
                dtype=np.int32,
            )

            overall_metrics = _cv_ranking_metrics(
                top10,
                overall_targets,
            )
            warm_metrics = _cv_ranking_metrics(
                top10,
                warm_targets,
            )
            cold_metrics = _cv_ranking_metrics(
                top10,
                cold_targets,
            )

            top10_segments = [
                factor_item_segment[item]
                for item in top10
            ]

            eval_rows.append({
                "fold_id": fold_id,
                "user_id": user_id,
                "profile_size": int(profile_size),
                "sampling_mode": sampling_mode,
                "repeat_id": int(repeat_id),
                "method": method_name,

                "n_overall_targets": len(overall_targets),
                "n_warm_tail_targets": len(warm_targets),
                "n_strict_cold_targets": len(cold_targets),

                "overall_ndcg": overall_metrics["ndcg"],
                "overall_recall": overall_metrics["recall"],
                "overall_hit": overall_metrics["hit"],

                "warm_tail_ndcg": warm_metrics["ndcg"],
                "warm_tail_recall": warm_metrics["recall"],
                "warm_tail_hit": warm_metrics["hit"],

                "strict_cold_recall": cold_metrics["recall"],

                "warm_tail_items_in_top10": (
                    top10_segments.count("warm-tail")
                ),
                "head_items_in_top10": (
                    top10_segments.count("head")
                ),
                "strict_cold_items_in_top10": (
                    top10_segments.count("strict-cold")
                ),
                "top10_item_indices": top10.tolist(),
            })

            for item_id in warm_targets:

                item_id = int(item_id)

                rank_rows.append({
                    "fold_id": fold_id,
                    "user_id": user_id,
                    "profile_size": int(profile_size),
                    "sampling_mode": sampling_mode,
                    "repeat_id": int(repeat_id),
                    "method": method_name,
                    "item_index": item_id,
                    "rank": int(rank_by_item[item_id]),
                    "rank_percentile": (
                        100.0 * (rank_by_item[item_id] - 1)
                        / (n_items - 1)
                    ),
                })

        if profile_number % 2000 == 0:
            print(
                f"  {profile_number:,}/"
                f"{len(support_samples):,} profiles processed"
            )

    fold_eval_df = pd.DataFrame(eval_rows)
    fold_rank_df = pd.DataFrame(rank_rows)

    assert not fold_eval_df.empty
    assert not fold_rank_df.empty
    assert fold_eval_df["fold_id"].nunique() == 1
    assert fold_rank_df["fold_id"].nunique() == 1

    factor_cv_eval_parts.append(fold_eval_df)
    factor_cv_rank_parts.append(fold_rank_df)

    factor_cv_fit_audit.append({
        "fold_id": fold_id,
        "als_training_seconds": als_fit["training_time_seconds"],
        "head_mapper_items": len(head_mapper_pool),
        "warm_tail_items": int(fold_ctx["warm_tail_mask"].sum()),
        "warm_tail_items_imputed": len(warm_tail_with_metadata),
        "warm_tail_items_without_metadata": len(
            warm_tail_without_metadata
        ),
        "sampled_profiles": len(support_samples),
        "evaluation_rows": len(fold_eval_df),
        "warm_tail_rank_rows": len(fold_rank_df),
        "total_fold_seconds": time.perf_counter() - fold_start,
    })

    print(
        f"\nFold {fold_id} complete | "
        f"{len(fold_eval_df):,} evaluation rows | "
        f"{len(fold_rank_df):,} warm-tail ranks | "
        f"{(time.perf_counter() - fold_start) / 60:.2f} min"
    )

    # Free large fold-specific objects.
    del (
        fold_ctx,
        als_fit,
        Q_als,
        Q_als_imputed,
        Q_warm_tail_pred,
        factor_mapper,
        mapper_validation,
        fold_eval_df,
        fold_rank_df,
        eval_rows,
        rank_rows,
    )
    gc.collect()


# ------------------------------------------------------------
# 7. Combine fold 0 with folds 1-4
# ------------------------------------------------------------

factor_cv_evaluation = pd.concat(
    factor_cv_eval_parts,
    ignore_index=True,
)

factor_cv_tail_ranks = pd.concat(
    factor_cv_rank_parts,
    ignore_index=True,
)

assert set(factor_cv_evaluation["fold_id"].unique()) == set(range(5))
assert set(factor_cv_tail_ranks["fold_id"].unique()) == set(range(5))

assert set(factor_cv_evaluation["method"].unique()) == {
    "ALS original",
    "ALS content-imputed",
}

print("\nFive-fold factor-imputation evaluation complete.")
print("Evaluation rows:", f"{len(factor_cv_evaluation):,}")
print("Warm-tail rank observations:", f"{len(factor_cv_tail_ranks):,}")

display(pd.DataFrame(factor_cv_mapper_diagnostics).round(5))
display(pd.DataFrame(factor_cv_fit_audit).round(2))



CONTENT-TO-ALS — FOLD 1
Fitting ALS for fold 1...
ALS iteration 01/10 | 7.8s
ALS iteration 02/10 | 13.2s
ALS iteration 04/10 | 23.4s
ALS iteration 06/10 | 30.6s
ALS iteration 08/10 | 38.1s
ALS iteration 10/10 | 45.3s

Held-out head-item factor diagnostics:


,value
RMSE,0.11703
normalized_RMSE,0.88618
variance_weighted_R2,0.14255
mean_cosine_similarity,0.58569
median_cosine_similarity,0.58428
median_predicted_to_true_norm_ratio,0.81287
zero_baseline_RMSE,0.13215
mean_baseline_RMSE,0.12644
fold_id,1.00000
head_items_with_metadata,5128.00000



Evaluating 13,812 profiles for fold 1...
  2,000/13,812 profiles processed
  4,000/13,812 profiles processed
  6,000/13,812 profiles processed
  8,000/13,812 profiles processed
  10,000/13,812 profiles processed
  12,000/13,812 profiles processed

Fold 1 complete | 27,624 evaluation rows | 9,984 warm-tail ranks | 2.82 min

CONTENT-TO-ALS — FOLD 2
Fitting ALS for fold 2...
ALS iteration 01/10 | 4.1s
ALS iteration 02/10 | 8.0s
ALS iteration 04/10 | 15.6s
ALS iteration 06/10 | 23.1s
ALS iteration 08/10 | 30.6s
ALS iteration 10/10 | 38.7s

Held-out head-item factor diagnostics:


,value
RMSE,0.10998
normalized_RMSE,0.87014
variance_weighted_R2,0.16832
mean_cosine_similarity,0.58986
median_cosine_similarity,0.59537
median_predicted_to_true_norm_ratio,0.81486
zero_baseline_RMSE,0.12646
mean_baseline_RMSE,0.12067
fold_id,2.00000
head_items_with_metadata,5091.00000



Evaluating 13,876 profiles for fold 2...
  2,000/13,876 profiles processed
  4,000/13,876 profiles processed
  6,000/13,876 profiles processed
  8,000/13,876 profiles processed
  10,000/13,876 profiles processed
  12,000/13,876 profiles processed

Fold 2 complete | 27,752 evaluation rows | 11,464 warm-tail ranks | 2.74 min

CONTENT-TO-ALS — FOLD 3
Fitting ALS for fold 3...
ALS iteration 01/10 | 3.8s
ALS iteration 02/10 | 9.2s
ALS iteration 04/10 | 16.8s
ALS iteration 06/10 | 24.7s
ALS iteration 08/10 | 33.1s
ALS iteration 10/10 | 40.7s

Held-out head-item factor diagnostics:


...


Evaluating 13,808 profiles for fold 3...
  2,000/13,808 profiles processed
  4,000/13,808 profiles processed
  6,000/13,808 profiles processed
  8,000/13,808 profiles processed
  10,000/13,808 profiles processed
  12,000/13,808 profiles processed

Fold 3 complete | 27,616 evaluation rows | 9,880 warm-tail ranks | 2.76 min

CONTENT-TO-ALS — FOLD 4
Fitting ALS for fold 4...
ALS iteration 01/10 | 3.7s
ALS iteration 02/10 | 7.7s
ALS iteration 04/10 | 15.2s
ALS iteration 06/10 | 22.7s
ALS iteration 08/10 | 30.2s
ALS iteration 10/10 | 37.6s

Held-out head-item factor diagnostics:


,value
RMSE,0.11813
normalized_RMSE,0.88582
variance_weighted_R2,0.14472
mean_cosine_similarity,0.58375
median_cosine_similarity,0.59331
median_predicted_to_true_norm_ratio,0.81125
zero_baseline_RMSE,0.13346
mean_baseline_RMSE,0.12779
fold_id,4.00000
head_items_with_metadata,5130.00000



Evaluating 13,872 profiles for fold 4...
  2,000/13,872 profiles processed
  4,000/13,872 profiles processed
  6,000/13,872 profiles processed
  8,000/13,872 profiles processed
  10,000/13,872 profiles processed
  12,000/13,872 profiles processed

Fold 4 complete | 27,744 evaluation rows | 9,720 warm-tail ranks | 2.63 min

Five-fold factor-imputation evaluation complete.
Evaluation rows: 138,344
Warm-tail rank observations: 49,888


,RMSE,normalized_RMSE,variance_weighted_R2,mean_cosine_similarity,median_cosine_similarity,median_predicted_to_true_norm_ratio,zero_baseline_RMSE,mean_baseline_RMSE,fold_id,head_items_with_metadata,mapper_train_items,mapper_validation_items
0,0.11703,0.88618,0.14255,0.58569,0.58428,0.81287,0.13215,0.12644,1,5128,4102,1026
1,0.10998,0.87014,0.16832,0.58986,0.59537,0.81486,0.12646,0.12067,2,5091,4072,1019
2,0.11772,0.88355,0.14751,0.58317,0.58897,0.78665,0.13328,0.12756,3,5139,4111,1028
3,0.11813,0.88582,0.14472,0.58375,0.59331,0.81125,0.13346,0.12779,4,5130,4104,1026


,fold_id,als_training_seconds,head_mapper_items,warm_tail_items,warm_tail_items_imputed,warm_tail_items_without_metadata,sampled_profiles,evaluation_rows,warm_tail_rank_rows,total_fold_seconds
0,1,45.34,5128,10461,10450,11,13812,27624,9984,169.43
1,2,38.69,5091,10430,10419,11,13876,27752,11464,164.41
2,3,40.70,5139,10491,10481,10,13808,27616,9880,165.44
3,4,37.64,5130,10384,10375,9,13872,27744,9720,157.76


In [19]:

# ============================================================
# FIVE-FOLD CONTENT-TO-ALS RESULTS
# ============================================================

# Average repeated support samples within each user first.
user_cols = [
    "fold_id",
    "user_id",
    "profile_size",
    "sampling_mode",
    "method",
]

factor_cv_user_level = (
    factor_cv_evaluation
    .groupby(user_cols, as_index=False)
    .agg(
        n_overall_targets=("n_overall_targets", "max"),
        n_warm_tail_targets=("n_warm_tail_targets", "max"),

        overall_ndcg=("overall_ndcg", "mean"),
        overall_recall=("overall_recall", "mean"),
        overall_hit_rate=("overall_hit", "mean"),

        warm_tail_recall=("warm_tail_recall", "mean"),
        warm_tail_hit_any_repeat=("warm_tail_hit", "max"),

        mean_warm_tail_items_in_top10=(
            "warm_tail_items_in_top10", "mean"
        ),
    )
)

factor_cv_user_level["has_warm_tail_target"] = (
    factor_cv_user_level["n_warm_tail_targets"] > 0
)

factor_cv_user_level["warm_tail_discovery"] = np.where(
    factor_cv_user_level["has_warm_tail_target"],
    factor_cv_user_level["warm_tail_hit_any_repeat"],
    np.nan,
)


def _factor_cv_summarize_users(data, group_cols):
    return (
        data.groupby(group_cols, as_index=False)
        .agg(
            users_evaluated=("user_id", "nunique"),

            overall_NDCG_at_10=("overall_ndcg", "mean"),
            overall_Recall_at_10=("overall_recall", "mean"),
            overall_HitRate_at_10=("overall_hit_rate", "mean"),

            warm_tail_target_users=("has_warm_tail_target", "sum"),
            warm_tail_Recall_at_10=("warm_tail_recall", "mean"),
            warm_tail_discovery_rate_any_repeat=(
                "warm_tail_discovery", "mean"
            ),
            mean_warm_tail_items_in_top10=(
                "mean_warm_tail_items_in_top10", "mean"
            ),
        )
    )


factor_cv_pooled_metrics = _factor_cv_summarize_users(
    factor_cv_user_level,
    ["profile_size", "sampling_mode", "method"],
)

factor_cv_fold_metrics = _factor_cv_summarize_users(
    factor_cv_user_level,
    ["fold_id", "profile_size", "sampling_mode", "method"],
)


# Full-catalogue warm-tail target rank summaries.
def _factor_cv_summarize_ranks(data, group_cols):
    return (
        data.groupby(group_cols, as_index=False)
        .agg(
            rank_observations=("rank", "size"),
            median_rank=("rank", "median"),
            mean_rank=("rank", "mean"),
            p90_rank=("rank", lambda x: x.quantile(0.90)),
            pct_targets_top10=("rank", lambda x: (x <= 10).mean()),
            pct_targets_top100=("rank", lambda x: (x <= 100).mean()),
            pct_targets_top500=("rank", lambda x: (x <= 500).mean()),
            pct_targets_top1000=("rank", lambda x: (x <= 1000).mean()),
        )
    )


factor_cv_pooled_ranks = _factor_cv_summarize_ranks(
    factor_cv_tail_ranks,
    ["profile_size", "sampling_mode", "method"],
)

factor_cv_fold_ranks = _factor_cv_summarize_ranks(
    factor_cv_tail_ranks,
    ["fold_id", "profile_size", "sampling_mode", "method"],
)


# Fold-level consistency.
rank500 = factor_cv_fold_ranks.pivot(
    index=["fold_id", "profile_size", "sampling_mode"],
    columns="method",
    values="pct_targets_top500",
)

rank1000 = factor_cv_fold_ranks.pivot(
    index=["fold_id", "profile_size", "sampling_mode"],
    columns="method",
    values="pct_targets_top1000",
)

median_rank = factor_cv_fold_ranks.pivot(
    index=["fold_id", "profile_size", "sampling_mode"],
    columns="method",
    values="median_rank",
)

ndcg = factor_cv_fold_metrics.pivot(
    index=["fold_id", "profile_size", "sampling_mode"],
    columns="method",
    values="overall_NDCG_at_10",
)

original = "ALS original"
imputed = "ALS content-imputed"

factor_cv_consistency = pd.DataFrame({
    "imputed_wins_top500": rank500[imputed] > rank500[original],
    "imputed_wins_top1000": rank1000[imputed] > rank1000[original],
    "imputed_has_better_median_rank": (
        median_rank[imputed] < median_rank[original]
    ),
    "imputed_wins_overall_ndcg": ndcg[imputed] > ndcg[original],
}).reset_index()

factor_cv_consistency_summary = (
    factor_cv_consistency
    .groupby(["profile_size", "sampling_mode"], as_index=False)
    .agg(
        folds_better_top500=("imputed_wins_top500", "sum"),
        folds_better_top1000=("imputed_wins_top1000", "sum"),
        folds_better_median_rank=(
            "imputed_has_better_median_rank", "sum"
        ),
        folds_better_overall_ndcg=("imputed_wins_overall_ndcg", "sum"),
        total_folds=("fold_id", "nunique"),
    )
)

print("A. POOLED FIVE-FOLD TOP-10 METRICS")
display(
    factor_cv_pooled_metrics.sort_values(
        ["profile_size", "sampling_mode", "method"]
    ).round(5)
)

print("\nB. POOLED WARM-TAIL RANKS")
display(
    factor_cv_pooled_ranks.sort_values(
        ["profile_size", "sampling_mode", "method"]
    ).round(5)
)

print("\nC. FOLD-WISE CONSISTENCY")
display(factor_cv_consistency_summary)

print("\nD. MAPPER VALIDATION DIAGNOSTICS")
display(
    pd.DataFrame(factor_cv_mapper_diagnostics)
    .sort_values("fold_id")
    .round(5)
)


A. POOLED FIVE-FOLD TOP-10 METRICS


,profile_size,sampling_mode,method,users_evaluated,overall_NDCG_at_10,overall_Recall_at_10,overall_HitRate_at_10,warm_tail_target_users,warm_tail_Recall_at_10,warm_tail_discovery_rate_any_repeat,mean_warm_tail_items_in_top10
0,3,popularity_biased,ALS content-imputed,4527,0.02352,0.03660,0.11012,929,0.0,0.0,0.0
1,3,popularity_biased,ALS original,4527,0.02352,0.03660,0.11012,929,0.0,0.0,0.0
2,3,uniform,ALS content-imputed,4527,0.02516,0.03854,0.11465,929,0.0,0.0,0.0
3,3,uniform,ALS original,4527,0.02516,0.03854,0.11465,929,0.0,0.0,0.0
4,5,popularity_biased,ALS content-imputed,4523,0.02563,0.04025,0.11596,928,0.0,0.0,0.0
5,5,popularity_biased,ALS original,4523,0.02563,0.04025,0.11596,928,0.0,0.0,0.0
6,5,uniform,ALS content-imputed,4523,0.02748,0.04182,0.12481,928,0.0,0.0,0.0
7,5,uniform,ALS original,4523,0.02748,0.04182,0.12481,928,0.0,0.0,0.0
8,10,popularity_biased,ALS content-imputed,4455,0.03006,0.04680,0.12997,922,0.0,0.0,0.0
9,10,popularity_biased,ALS original,4455,0.03006,0.04680,0.12997,922,0.0,0.0,0.0



B. POOLED WARM-TAIL RANKS


,profile_size,sampling_mode,method,rank_observations,median_rank,mean_rank,p90_rank,pct_targets_top10,pct_targets_top100,pct_targets_top500,pct_targets_top1000
0,3,popularity_biased,ALS content-imputed,3158,4610.5,5892.26916,13808.5,0.0,0.0,0.03673,0.11305
1,3,popularity_biased,ALS original,3158,5427.0,6961.59721,14423.6,0.0,0.0,0.00095,0.03135
2,3,uniform,ALS content-imputed,3158,3898.0,5164.18493,12335.0,0.0,0.0,0.04307,0.14281
3,3,uniform,ALS original,3158,4716.0,6262.20773,13972.2,0.0,0.0,0.00443,0.04940
4,5,popularity_biased,ALS content-imputed,3156,4031.5,5378.95596,12855.0,0.0,0.0,0.03739,0.12643
5,5,popularity_biased,ALS original,3156,5252.5,6753.48194,14264.5,0.0,0.0,0.00222,0.02788
6,5,uniform,ALS content-imputed,3156,3444.0,4807.66191,11364.0,0.0,0.0,0.05228,0.16001
7,5,uniform,ALS original,3156,4184.0,5680.91857,13348.0,0.0,0.0,0.00412,0.04658
8,10,popularity_biased,ALS content-imputed,3142,3629.5,5021.46976,11731.4,0.0,0.0,0.04615,0.15532
9,10,popularity_biased,ALS original,3142,4549.5,6144.64863,13909.9,0.0,0.0,0.00064,0.03183



C. FOLD-WISE CONSISTENCY


,profile_size,sampling_mode,folds_better_top500,folds_better_top1000,folds_better_median_rank,folds_better_overall_ndcg,total_folds
0,3,popularity_biased,5,5,5,0,5
1,3,uniform,5,5,4,0,5
2,5,popularity_biased,5,5,5,0,5
3,5,uniform,5,5,5,0,5
4,10,popularity_biased,5,5,5,0,5
5,10,uniform,5,5,5,0,5
6,20,popularity_biased,5,5,4,0,5
7,20,uniform,5,5,4,0,5



D. MAPPER VALIDATION DIAGNOSTICS


,RMSE,normalized_RMSE,variance_weighted_R2,mean_cosine_similarity,median_cosine_similarity,median_predicted_to_true_norm_ratio,zero_baseline_RMSE,mean_baseline_RMSE,fold_id,head_items_with_metadata,mapper_train_items,mapper_validation_items
0,0.11703,0.88618,0.14255,0.58569,0.58428,0.81287,0.13215,0.12644,1,5128,4102,1026
1,0.10998,0.87014,0.16832,0.58986,0.59537,0.81486,0.12646,0.12067,2,5091,4072,1019
2,0.11772,0.88355,0.14751,0.58317,0.58897,0.78665,0.13328,0.12756,3,5139,4111,1028
3,0.11813,0.88582,0.14472,0.58375,0.59331,0.81125,0.13346,0.12779,4,5130,4104,1026


In [21]:

# ============================================================
# FOLD 0 — TAIL-ONLY AUC: CONTENT SIGNAL VS CONTROLS
# ============================================================

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score
import time

# ------------------------------------------------------------
# 0. Validate existing fold-0 objects
# ------------------------------------------------------------

assert Q_als_smoke.shape == (n_items, ALS_FACTORS)
assert Q_als_imputed_fold0.shape == Q_als_smoke.shape
assert X_factor.shape == (n_items, 31)
assert has_metadata.shape == (n_items,)

assert np.array_equal(
    Q_als_smoke, als_smoke["Q"]
), "Original ALS factors have been modified."

# Restrict every method to the SAME warm-tail candidate pool.
# Items without metadata cannot receive a content prediction,
# so they are excluded from all arms in this diagnostic.

tail_item_ids = np.flatnonzero(
    smoke_fold["warm_tail_mask"] & has_metadata
)

head_item_ids = np.flatnonzero(
    smoke_fold["head_mask"] & has_metadata
)

assert len(tail_item_ids) > 0
assert len(head_item_ids) > X_factor.shape[1]

# Confirm the fold-0 item set independently.
assert set(tail_item_ids) == set(
    np.flatnonzero(
        smoke_fold["warm_tail_mask"] & has_metadata
    )
)

print("Fold-0 warm-tail candidate count:", len(tail_item_ids))
print(
    "Stale warm_tail_with_metadata count:",
    len(warm_tail_with_metadata)
)

Q_original_tail = Q_als_smoke[tail_item_ids].astype(np.float32)
Q_imputed_tail = Q_als_imputed_fold0[tail_item_ids].astype(np.float32)

assert np.isfinite(Q_original_tail).all()
assert np.isfinite(Q_imputed_tail).all()

print("Warm-tail candidates with metadata:", len(tail_item_ids))
print("Head items used for controls:", len(head_item_ids))


# ------------------------------------------------------------
# 1. Norm-matched original ALS control
# ------------------------------------------------------------

# Retain original ALS directions, but give every item the
# corresponding imputed factor's norm.
#
# This isolates whether the content mapper contributes useful
# directional information beyond its factor-magnitude changes.

original_norms = np.linalg.norm(
    Q_original_tail.astype(np.float64),
    axis=1,
)

imputed_norms = np.linalg.norm(
    Q_imputed_tail.astype(np.float64),
    axis=1,
)

if np.any(original_norms <= 1e-12):
    raise ValueError(
        "Some warm-tail original ALS factors have zero norm. "
        "Inspect these items before norm matching."
    )

Q_norm_matched_tail = (
    Q_original_tail.astype(np.float64)
    * (imputed_norms / original_norms)[:, None]
).astype(np.float32)

assert np.allclose(
    np.linalg.norm(Q_norm_matched_tail, axis=1),
    imputed_norms,
    rtol=1e-5,
    atol=1e-6,
)

print(
    "Median original tail norm:",
    round(float(np.median(original_norms)), 5),
)
print(
    "Median imputed tail norm:",
    round(float(np.median(imputed_norms)), 5),
)
print(
    "Median norm-matched tail norm:",
    round(
        float(np.median(np.linalg.norm(Q_norm_matched_tail, axis=1))),
        5,
    ),
)


# ------------------------------------------------------------
# 2. Mean-head-factor control
# ------------------------------------------------------------

# Every eligible warm-tail item receives the same factor.
# Therefore every candidate receives the same score for a
# given user. With tie-aware AUC, this should be approximately 0.5.

mean_head_factor = Q_als_smoke[head_item_ids].mean(
    axis=0
).astype(np.float32)

assert mean_head_factor.shape == (ALS_FACTORS,)


# ------------------------------------------------------------
# 3. Shuffled-feature mapper controls
# ------------------------------------------------------------

# Pair each head item's ALS factor with another head item's
# metadata feature row. The true feature-to-factor relationship
# is destroyed, while the feature and target distributions remain.

SHUFFLE_SEEDS = [
    SEED + 901,
    SEED + 902,
    SEED + 903,
]

shuffled_tail_predictions = {}
shuffled_mapper_diagnostics = []

shuffle_start = time.perf_counter()

X_head = X_factor[head_item_ids]
Y_head = Q_als_smoke[head_item_ids]

for shuffle_number, shuffle_seed in enumerate(
    SHUFFLE_SEEDS,
    start=1,
):

    rng = np.random.default_rng(shuffle_seed)

    permutation = rng.permutation(len(head_item_ids))

    shuffled_model = Ridge(
        alpha=FACTOR_RIDGE_ALPHA,
        fit_intercept=True,
        solver="lsqr",
    )

    # Deliberately mismatch feature rows and factor targets.
    shuffled_model.fit(
        X_head[permutation],
        Y_head,
    )

    shuffled_prediction = shuffled_model.predict(
        X_factor[tail_item_ids]
    ).astype(np.float32)

    assert shuffled_prediction.shape == Q_imputed_tail.shape
    assert np.isfinite(shuffled_prediction).all()

    method_name = f"Shuffled features {shuffle_number}"

    shuffled_tail_predictions[method_name] = shuffled_prediction

    shuffled_mapper_diagnostics.append({
        "method": method_name,
        "seed": shuffle_seed,
        "training_items": len(head_item_ids),
        "prediction_items": len(tail_item_ids),
    })

print(
    f"\nThree shuffled mappers fitted in "
    f"{time.perf_counter() - shuffle_start:.2f}s"
)

display(pd.DataFrame(shuffled_mapper_diagnostics))


# ------------------------------------------------------------
# 4. Stack candidate factor matrices
# ------------------------------------------------------------

factor_methods = {
    "ALS original": Q_original_tail,
    "Content imputed": Q_imputed_tail,
    "Norm-matched ALS": Q_norm_matched_tail,
}

factor_methods.update(shuffled_tail_predictions)

factor_method_names = list(factor_methods.keys())

# Shape: (number of methods, number of tail candidates, factors)
candidate_factor_stack = np.stack(
    [factor_methods[name] for name in factor_method_names],
    axis=0,
).astype(np.float32)

assert candidate_factor_stack.shape == (
    len(factor_method_names),
    len(tail_item_ids),
    ALS_FACTORS,
)

# User-independent popularity baseline.
tail_popularity = smoke_fold["popularity_counts"][
    tail_item_ids
].astype(np.float64)


# ------------------------------------------------------------
# 5. Tie audit for metadata rows
# ------------------------------------------------------------

# Linear Ridge prediction gives identical factor predictions
# to identical feature rows. Count these explicitly.

unique_tail_feature_rows, feature_row_counts = np.unique(
    X_factor[tail_item_ids],
    axis=0,
    return_counts=True,
)

duplicate_feature_groups = feature_row_counts[
    feature_row_counts > 1
]

feature_tie_audit = pd.Series({
    "warm_tail_candidates": len(tail_item_ids),
    "unique_metadata_feature_rows": len(unique_tail_feature_rows),
    "extra_duplicate_rows": (
        len(tail_item_ids) - len(unique_tail_feature_rows)
    ),
    "number_of_repeated_feature_groups": len(duplicate_feature_groups),
    "items_in_repeated_feature_groups": int(
        duplicate_feature_groups.sum()
    ),
    "largest_duplicate_feature_group": int(
        feature_row_counts.max()
    ),
    "fraction_of_items_in_repeated_groups": (
        float(duplicate_feature_groups.sum()) / len(tail_item_ids)
        if len(duplicate_feature_groups) else 0.0
    ),
})

print("\nA. METADATA FEATURE DUPLICATION")
display(feature_tie_audit.to_frame("value"))


# ------------------------------------------------------------
# 6. Evaluate tail-only ROC AUC per support profile
# ------------------------------------------------------------

# Each profile:
# - Uses the same original ALS fold-in vector for every method.
# - Ranks only metadata-covered warm-tail candidates.
# - Excludes the selected support movies.
# - Treats warm-tail validation targets as positives.
#
# Profiles without an eligible warm-tail target are skipped.
# Users, not individual target pairs, will be the averaging unit.

auc_rows = []
target_coverage_rows = []
tie_rows = []

TIE_AUDIT_EVERY = 100
start = time.perf_counter()

support_samples = smoke_fold["support_samples"]

print(
    "\nProfiles available:",
    f"{len(support_samples):,}"
)

for profile_number, (key, profile_items) in enumerate(
    support_samples.items(),
    start=1,
):

    user_id, profile_size, sampling_mode, repeat_id = key

    user_id = int(user_id)
    profile_items = np.asarray(
        profile_items,
        dtype=np.int64,
    )

    warm_targets = smoke_fold["warm_tail_relevant"].get(
        user_id, set()
    )

    if not warm_targets:
        continue

    # Determine which warm-tail targets are eligible for this
    # metadata-restricted comparison.
    target_array = np.asarray(
        sorted(warm_targets),
        dtype=np.int64,
    )

    eligible_target_items = np.intersect1d(
        target_array,
        tail_item_ids,
        assume_unique=False,
    )

    target_coverage_rows.append({
        "user_id": user_id,
        "profile_size": int(profile_size),
        "sampling_mode": sampling_mode,
        "repeat_id": int(repeat_id),
        "all_warm_tail_targets": len(target_array),
        "metadata_covered_targets": len(eligible_target_items),
    })

    if len(eligible_target_items) == 0:
        continue

    # Remove selected support items from the candidate pool.
    candidate_mask = ~np.isin(
        tail_item_ids,
        profile_items,
    )

    candidate_local_indices = np.flatnonzero(candidate_mask)

    candidate_item_ids = tail_item_ids[
        candidate_local_indices
    ]

    positive_mask = np.isin(
        candidate_item_ids,
        eligible_target_items,
    )

    n_positive = int(positive_mask.sum())
    n_negative = int((~positive_mask).sum())

    if n_positive == 0 or n_negative == 0:
        continue

    # Crucial: same original ALS item matrix for fold-in.
    user_vector = fold_in_als_profile(
        profile_items,
        Q_als_smoke,
    )

    # Score all factor-based methods with the SAME user vector.
    factor_scores = np.einsum(
        "mjk,k->mj",
        candidate_factor_stack,
        user_vector,
        optimize=True,
    ).astype(np.float64)

    factor_scores = factor_scores[:, candidate_local_indices]

    method_scores = {
        name: factor_scores[index]
        for index, name in enumerate(factor_method_names)
    }

    # Mean-head factor: all remaining tail candidates tie.
    mean_score = float(user_vector @ mean_head_factor)

    method_scores["Mean head factor"] = np.full(
        len(candidate_item_ids),
        mean_score,
        dtype=np.float64,
    )

    # Popularity: same fold-specific count for all users.
    method_scores["Popularity"] = tail_popularity[
        candidate_local_indices
    ]

    # Evaluate tie-aware AUC.
    for method_name, scores in method_scores.items():

        auc = roc_auc_score(
            positive_mask.astype(np.int8),
            scores,
        )

        auc_rows.append({
            "user_id": user_id,
            "profile_size": int(profile_size),
            "sampling_mode": sampling_mode,
            "repeat_id": int(repeat_id),
            "method": method_name,
            "warm_tail_auc": float(auc),
            "n_positive_targets": n_positive,
            "n_negative_candidates": n_negative,
            "candidate_count": len(candidate_item_ids),
        })

    # Exact score tie diagnostics on a small deterministic
    # sample of profiles. This avoids sorting thousands of
    # candidate scores for every profile.
    if profile_number % TIE_AUDIT_EVERY == 0:

        for method_name, scores in method_scores.items():

            _, counts = np.unique(
                scores,
                return_counts=True,
            )

            tie_rows.append({
                "user_id": user_id,
                "profile_size": int(profile_size),
                "sampling_mode": sampling_mode,
                "method": method_name,
                "unique_scores": len(counts),
                "largest_exact_tie_block": int(counts.max()),
                "candidate_count": len(scores),
            })

    if profile_number % 2000 == 0:
        print(
            f"Processed {profile_number:,}/"
            f"{len(support_samples):,} profiles"
        )


factor_tail_auc_profiles = pd.DataFrame(auc_rows)
factor_tail_target_coverage = pd.DataFrame(target_coverage_rows)
factor_tail_tie_profiles = pd.DataFrame(tie_rows)

assert not factor_tail_auc_profiles.empty
assert factor_tail_auc_profiles["warm_tail_auc"].between(0, 1).all()

print(
    "\nTail-only AUC evaluation completed in",
    f"{(time.perf_counter() - start) / 60:.2f} minutes",
)

print(
    "Evaluated profile-method rows:",
    f"{len(factor_tail_auc_profiles):,}",
)

print(
    "Unique users with eligible warm-tail targets:",
    factor_tail_auc_profiles["user_id"].nunique(),
)


# ------------------------------------------------------------
# 7. Average repeated samples within each user
# ------------------------------------------------------------

user_group_cols = [
    "user_id",
    "profile_size",
    "sampling_mode",
    "method",
]

factor_tail_auc_users = (
    factor_tail_auc_profiles
    .groupby(user_group_cols, as_index=False)
    .agg(
        mean_auc=("warm_tail_auc", "mean"),
        profile_repeats=("repeat_id", "nunique"),
        mean_positive_targets=("n_positive_targets", "mean"),
    )
)


# Create a per-user average of the three shuffled controls.
shuffle_user_mean = (
    factor_tail_auc_users[
        factor_tail_auc_users["method"].str.startswith(
            "Shuffled features"
        )
    ]
    .groupby(
        ["user_id", "profile_size", "sampling_mode"],
        as_index=False,
    )
    .agg(
        mean_auc=("mean_auc", "mean"),
        profile_repeats=("profile_repeats", "max"),
        mean_positive_targets=("mean_positive_targets", "mean"),
    )
)

shuffle_user_mean["method"] = (
    "Shuffled features (3-run mean)"
)

factor_tail_auc_users = pd.concat(
    [
        factor_tail_auc_users,
        shuffle_user_mean,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 8. Macro AUC summary
# ------------------------------------------------------------

factor_tail_auc_summary = (
    factor_tail_auc_users
    .groupby(
        ["profile_size", "sampling_mode", "method"],
        as_index=False,
    )
    .agg(
        users=("user_id", "nunique"),
        macro_mean_auc=("mean_auc", "mean"),
        median_user_auc=("mean_auc", "median"),
        std_user_auc=("mean_auc", "std"),
    )
)

factor_tail_auc_summary["random_auc_reference"] = 0.5

print("\nB. MACRO USER-LEVEL TAIL AUC")
display(
    factor_tail_auc_summary
    .sort_values(
        ["profile_size", "sampling_mode", "method"]
    )
    .round(5)
)


# ------------------------------------------------------------
# 9. Paired user-level bootstrap CIs
# ------------------------------------------------------------

# Repeated support samples have already been averaged within
# each user. Bootstrap resampling is now over users.
#
# Positive delta means content-imputed AUC is higher.

BOOTSTRAP_REPEATS = 2000

baseline_methods = [
    "ALS original",
    "Mean head factor",
    "Popularity",
    "Norm-matched ALS",
    "Shuffled features (3-run mean)",
]

paired_ci_rows = []

for (profile_size, sampling_mode), group in (
    factor_tail_auc_users.groupby(
        ["profile_size", "sampling_mode"]
    )
):

    wide = group.pivot(
        index="user_id",
        columns="method",
        values="mean_auc",
    )

    for baseline in baseline_methods:

        required = [
            "Content imputed",
            baseline,
        ]

        paired = wide[required].dropna()

        if len(paired) < 2:
            continue

        differences = (
            paired["Content imputed"].to_numpy()
            - paired[baseline].to_numpy()
        )

        rng = np.random.default_rng(
            SEED
            + int(profile_size) * 1000
            + (0 if sampling_mode == "uniform" else 100)
            + baseline_methods.index(baseline)
        )

        bootstrap_indices = rng.integers(
            0,
            len(differences),
            size=(BOOTSTRAP_REPEATS, len(differences)),
        )

        bootstrap_means = differences[
            bootstrap_indices
        ].mean(axis=1)

        ci_low, ci_high = np.quantile(
            bootstrap_means,
            [0.025, 0.975],
        )

        paired_ci_rows.append({
            "profile_size": int(profile_size),
            "sampling_mode": sampling_mode,
            "comparison": f"Content imputed - {baseline}",
            "paired_users": len(paired),
            "mean_auc_difference": float(differences.mean()),
            "ci_95_low": float(ci_low),
            "ci_95_high": float(ci_high),
        })


factor_tail_auc_paired_ci = pd.DataFrame(
    paired_ci_rows
)

print("\nC. PAIRED USER-LEVEL AUC DIFFERENCES")
display(
    factor_tail_auc_paired_ci
    .sort_values(
        ["profile_size", "sampling_mode", "comparison"]
    )
    .round(5)
)


# ------------------------------------------------------------
# 10. Target metadata coverage
# ------------------------------------------------------------

# Deduplicate repeats so target coverage is not multiplied by
# the two support-profile repeats.

coverage_users = (
    factor_tail_target_coverage
    .drop_duplicates(
        ["user_id", "profile_size", "sampling_mode"]
    )
)

coverage_summary = (
    coverage_users
    .groupby(
        ["profile_size", "sampling_mode"],
        as_index=False,
    )
    .agg(
        users=("user_id", "nunique"),
        all_warm_tail_targets=("all_warm_tail_targets", "sum"),
        metadata_covered_targets=("metadata_covered_targets", "sum"),
    )
)

coverage_summary["metadata_target_coverage"] = (
    coverage_summary["metadata_covered_targets"]
    / coverage_summary["all_warm_tail_targets"]
)

print("\nD. WARM-TAIL TARGET COVERAGE")
display(coverage_summary.round(5))


# ------------------------------------------------------------
# 11. Exact score tie-block audit
# ------------------------------------------------------------

if not factor_tail_tie_profiles.empty:

    factor_tail_tie_summary = (
        factor_tail_tie_profiles
        .groupby(
            ["profile_size", "sampling_mode", "method"],
            as_index=False,
        )
        .agg(
            sampled_profiles=("user_id", "size"),
            median_unique_scores=("unique_scores", "median"),
            median_largest_tie_block=(
                "largest_exact_tie_block", "median"
            ),
            p95_largest_tie_block=(
                "largest_exact_tie_block",
                lambda x: x.quantile(0.95),
            ),
        )
    )

    print("\nE. EXACT SCORE TIES ON SAMPLED PROFILES")
    display(
        factor_tail_tie_summary
        .sort_values(
            ["profile_size", "sampling_mode", "method"]
        )
        .round(3)
    )

print("\nFold-0 tail-only AUC experiment complete.")
print("No ALS factors or deployed artifacts were modified.")


Fold-0 warm-tail candidate count: 10458
Stale warm_tail_with_metadata count: 10375
Warm-tail candidates with metadata: 10458
Head items used for controls: 5166
Median original tail norm: 0.1005
Median imputed tail norm: 0.45518
Median norm-matched tail norm: 0.45518

Three shuffled mappers fitted in 0.43s


,method,seed,training_items,prediction_items
0,Shuffled features 1,943,5166,10458
1,Shuffled features 2,944,5166,10458
2,Shuffled features 3,945,5166,10458



A. METADATA FEATURE DUPLICATION


,value
warm_tail_candidates,10458.000000
unique_metadata_feature_rows,2118.000000
extra_duplicate_rows,8340.000000
number_of_repeated_feature_groups,891.000000
items_in_repeated_feature_groups,9231.000000
largest_duplicate_feature_group,433.000000
fraction_of_items_in_repeated_groups,0.882674



Profiles available: 13,804
Processed 4,000/13,804 profiles

Tail-only AUC evaluation completed in 1.96 minutes
Evaluated profile-method rows: 22,336
Unique users with eligible warm-tail targets: 178

B. MACRO USER-LEVEL TAIL AUC


...


D. WARM-TAIL TARGET COVERAGE


,profile_size,sampling_mode,users,all_warm_tail_targets,metadata_covered_targets,metadata_target_coverage
0,3,popularity_biased,178,281,281,1.0
1,3,uniform,178,281,281,1.0
2,5,popularity_biased,178,281,281,1.0
3,5,uniform,178,281,281,1.0
4,10,popularity_biased,178,281,281,1.0
5,10,uniform,178,281,281,1.0
6,20,popularity_biased,164,262,262,1.0
7,20,uniform,164,262,262,1.0



E. EXACT SCORE TIES ON SAMPLED PROFILES


,profile_size,sampling_mode,method,sampled_profiles,median_unique_scores,median_largest_tie_block,p95_largest_tie_block
0,3,popularity_biased,ALS original,6,9235.5,37.0,37.00
1,3,popularity_biased,Content imputed,6,2319.5,405.5,432.75
2,3,popularity_biased,Mean head factor,6,1.0,10458.0,10458.00
3,3,popularity_biased,Norm-matched ALS,6,10310.5,6.0,6.00
4,3,popularity_biased,Popularity,6,173.0,870.0,870.00
5,3,popularity_biased,Shuffled features 1,6,2199.5,432.0,432.75
6,3,popularity_biased,Shuffled features 2,6,2192.0,433.0,433.00
7,3,popularity_biased,Shuffled features 3,6,2223.0,432.5,433.00
8,5,popularity_biased,ALS original,6,9235.0,37.0,37.00
9,5,popularity_biased,Content imputed,6,2296.5,432.5,433.00



Fold-0 tail-only AUC experiment complete.
No ALS factors or deployed artifacts were modified.
